# Nova Retail – AI Operations Manager

A small **multi-agent AI system** for a fictional shop called *Nova Retail*. You can ask it things like
*"Which products might run out of stock?"* and several specialist programs ("agents") cooperate to answer.

## The big picture (read this first)
```
your question
   │
   ▼
 ROUTER ──► decides which agents are needed
   │
   ▼
 Agent 1  Data Collector   loads + checks the data files
   │
   ├─► Agent 2  Sales Analyst        (revenue, best sellers)
   ├─► Agent 3  Inventory Analyst    (what runs out, and when)      ← these run in parallel
   ├─► Agent 4  Feedback Analyst     (what customers complain about)
   └─► Agent 5  Document Researcher  (reads supplier/policy PDFs)
   │
   ▼
 Agent 6  Coordinator   combines findings into prioritised actions
   ▼
 Agent 7  Verifier      re-checks every number and source (never trusts the AI blindly)
   ▼
 Agent 8  Reporter      saves reports + proposed actions (a human approves them)
   ▼
 answer
```

## Two golden rules
1. **Python does the maths. The AI (Qwen) only explains in words.** Numbers always come from code, so they can be checked.
2. **Nothing is ever sent or ordered automatically.** The system only *proposes* actions and drafts; you approve or reject.

## How to use this notebook
* Run the cells **from top to bottom** (Runtime ▸ Run all also works). Each code cell has a text cell above it explaining it.
* Use a **GPU** (Runtime ▸ Change runtime type ▸ T4 GPU) so the AI model is fast. No GPU? It still works — it just uses plain templates instead of the AI for wording.
* No API keys, no paid services. The AI model runs inside Colab.

## Words you will see
| Word | Meaning |
|---|---|
| **LLM** | The language model (here: Qwen2.5-3B). Writes text. |
| **RAG** | "Retrieval-Augmented Generation": find the relevant passage in the PDFs first, then answer *only* from it. |
| **Embedding / FAISS** | Turning text into numbers so similar texts can be found quickly. |
| **LangGraph** | A library that runs the agents as a flow-chart (a "graph"). |
| **Coverage (days)** | How many days current stock will last at the recent selling speed. |
| **Lead time** | How many days a supplier needs to deliver after you order. |
| **SQLite** | A tiny database stored in one file. We use it to remember actions and reports. |

## Step 1 · Install the libraries
**What it does:** installs the extra Python packages we need. Colab already has PyTorch, so we do *not* reinstall it (that could break things).

**You should see:** a short list of installed packages (or nothing, if they are already there). Takes ~1 minute the first time.

In [ ]:
# Install only what Colab does not already have.
!pip install -q transformers accelerate sentence-transformers faiss-cpu langchain-core langchain-text-splitters langgraph pypdf reportlab gradio plotly pydantic openpyxl bitsandbytes

## Step 2 · Settings
**What it does:** three on/off switches you can change before running everything.

| Setting | Meaning |
|---|---|
| `USE_LLM` | `True` = load the Qwen AI model. `False` = skip it (fast, plain wording, good for testing). |
| `LOAD_IN_4BIT` | `True` = use less GPU memory (needs a GPU). Turn on if you get "out of memory". |
| `SHARE_GRADIO` | `True` creates a **public link** to the app. Leave `False` unless you really need it, and never use it with private data. |
| `USE_NARRATION` | `True` = the AI writes the short summaries. `False` = use the verified template text (instant). |
| `FEEDBACK_USE_LLM` | `True` = the AI classifies customer comments (slow the first time, cached afterwards). `False` = keyword classifier only (instant). |

**Tip:** the shop name, currency and risk thresholds are also here. Change them and re-run to see the effect.

In [ ]:
USE_LLM = True
LOAD_IN_4BIT = False
SHARE_GRADIO = True
USE_NARRATION = True      # False = skip AI wording, use the verified templates (instant)
FEEDBACK_USE_LLM = True   # False = keyword classifier only (instant); True = AI classifies each comment once, then caches it

# --- business settings (change freely) ---
BUSINESS = "Nova Retail"
CURRENCY = "$"
LOOKBACK_DAYS = 30        # days of sales used to estimate how fast a product sells
SAFETY_DAYS = 3           # extra buffer added to the supplier's lead time
TARGET_COVER_DAYS = 14    # when reordering, aim for this many days of stock after delivery
RISING_PCT = 15           # demand growth (%) that counts as "rising"
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

## Step 3 · Imports and folders
**What it does:** loads the standard tools and creates a working folder. In Colab the folder is `/content/nova_simple`.

⚠️ Files in `/content` are **deleted when the Colab session resets**. To keep them, mount Google Drive and change `WORK` to a Drive folder.

**You should see:** `Working folder: ...`

In [ ]:
import os, re, json, math, sqlite3, uuid, time, hashlib, threading, operator
from collections import Counter
from typing import Annotated, TypedDict, Literal
import numpy as np
import pandas as pd

WORK = "/content/nova_simple" if os.path.isdir("/content") else "nova_simple"
os.makedirs(WORK, exist_ok=True)
DATA = f"{WORK}/data"; os.makedirs(DATA, exist_ok=True)
DB_PATH = f"{WORK}/nova.sqlite"
print("Working folder:", os.path.abspath(WORK))

## Step 4 · Create the sample data
**What it does:** invents a small, realistic dataset so you can try everything without real business data:

| File | Contents |
|---|---|
| `sales.csv` | 61 days of order lines (`order_id, date, product_id, product_name, quantity, unit_price`) |
| `inventory.csv` | stock, reorder level, cost, supplier and lead time for 8 products |
| `customer_feedback.csv` | ~100 customer comments with ratings |
| `business_policy.pdf`, `supplier_terms.pdf` | fictional rules and supplier terms (used by the Document Researcher) |

The data is **designed** so the system has something to find: USB-C Hub demand is rising while stock is low, the Office Chair is almost out, the Desk Lamp gets quality complaints, and so on. Everything is random-but-repeatable (`seed=42`), so you get the same data every time.

**You should see:** a list of the created files.

In [ ]:
# (product_id, name, price, cost, supplier, lead_time_days, avg_sales_per_day, demand_trend, stock, reorder_level)
# demand_trend 1.7 means "selling 70% faster by the end of the period"
PRODUCTS = [
    ("P001", "Wireless Mouse",         24.99, 11.0, "Nile Tech Distribution", 14, 5.0, 1.0, 75, 60),
    ("P002", "USB-C Hub",              39.99, 18.0, "Nile Tech Distribution", 14, 4.0, 1.7, 45, 50),
    ("P003", "LED Desk Lamp",          29.99, 13.0, "Delta Home Supplies",    10, 3.0, 1.0, 33, 30),
    ("P004", "Ergonomic Office Chair", 149.99, 82.0, "Delta Home Supplies",   28, 0.7, 1.0, 2, 5),
    ("P005", "Notebook 3-Pack",        9.99, 3.5,  "BrightSource Wholesale",   7, 9.0, 1.0, 420, 120),
    ("P006", "Gel Pen Set",            6.49, 2.0,  "BrightSource Wholesale",   7, 8.0, 1.0, 40, 90),
    ("P007", "Reusable Water Bottle",  14.99, 5.5, "GreenLeaf Goods",         12, 4.0, 1.0, 70, 80),
    ("P008", "Backpack",               44.99, 20.0, "Delta Home Supplies",    10, 1.8, 1.0, 90, 20),
]
FB_BANK = {
    "delivery": ["Parcel arrived late, almost two weeks after the promised date.", "Delivery took far too long and tracking never updated.", "The courier delayed my order by a week."],
    "quality": ["Broke after a few days of normal use.", "Feels flimsy and cheap, poor build quality.", "Faulty unit, the light flickers constantly."],
    "price": ["Too expensive for what you get.", "Overpriced compared with other shops."],
    "availability": ["Always out of stock when I try to order.", "Sold out again, no restock date given.", "Unavailable for weeks now."],
    "customer_service": ["Support team ignored my refund request.", "Staff were rude when I asked about a return."],
    "praise": ["Great product, works exactly as described.", "Excellent value, very happy.", "Really happy with this purchase.", "Solid quality, would recommend."],
}
COMPLAINTS = {"P001": ["delivery"], "P002": ["availability", "delivery"], "P003": ["quality"],
              "P004": ["availability", "price"], "P006": ["availability"], "P007": ["customer_service"]}

def make_sample_data(folder=DATA, seed=42):
    rng = np.random.default_rng(seed)
    days = pd.date_range("2026-08-01", "2026-09-30")
    rows, oid = [], 1000
    for i, d in enumerate(days):
        lines = []                                    # all sale lines for this day
        for pid, name, price, cost, supplier, lead, base, trend, stock, reorder in PRODUCTS:
            units = rng.poisson(base * (1 + (trend - 1) * i / (len(days) - 1)))
            while units > 0:
                q = int(min(units, rng.choice([1, 1, 2]))); units -= q
                lines.append((pid, name, q, price))
        rng.shuffle(lines)
        order, seen = [], set()                       # group lines into orders of up to 3 different products
        for ln in lines + [None]:
            if ln is None or len(order) == 3 or ln[0] in seen:
                if order:
                    oid += 1
                    rows += [(f"ORD-{oid}", d.date().isoformat(), p, n, q, pr) for p, n, q, pr in order]
                order, seen = [], set()
            if ln is not None:
                order.append(ln); seen.add(ln[0])
    pd.DataFrame(rows, columns=["order_id", "date", "product_id", "product_name", "quantity", "unit_price"]).to_csv(f"{folder}/sales.csv", index=False)
    pd.DataFrame([(p[0], p[1], p[8], p[9], p[3], p[4], p[5]) for p in PRODUCTS],
                 columns=["product_id", "product_name", "stock_quantity", "reorder_level", "unit_cost", "supplier", "lead_time_days"]).to_csv(f"{folder}/inventory.csv", index=False)
    weights = np.array([p[6] for p in PRODUCTS]); weights = weights / weights.sum()
    fb = []
    for k in range(100):
        pid = str(rng.choice([p[0] for p in PRODUCTS], p=weights))
        if pid in COMPLAINTS and rng.random() < 0.55:
            text = str(rng.choice(FB_BANK[str(rng.choice(COMPLAINTS[pid]))])); rating = int(rng.choice([1, 2, 2, 3]))
        else:
            text = str(rng.choice(FB_BANK["praise"])); rating = int(rng.choice([4, 5, 5]))
        fb.append((f"F{k+1:03d}", days[int(rng.integers(0, len(days)))].date().isoformat(), rating, text, pid))
    pd.DataFrame(fb, columns=["feedback_id", "date", "rating", "feedback_text", "product_id"]).to_csv(f"{folder}/customer_feedback.csv", index=False)

    # two small PDFs (one page per heading) that the Document Researcher will read
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
    st = getSampleStyleSheet()
    def pdf(name, pages):
        story = []
        for i, (title, paras) in enumerate(pages):
            story += [Paragraph(title, st["Heading2"]), Spacer(1, 12)] + [x for t in paras for x in (Paragraph(t, st["BodyText"]), Spacer(1, 8))]
            if i < len(pages) - 1: story.append(PageBreak())
        SimpleDocTemplate(f"{folder}/{name}", pagesize=A4).build(story)
    pdf("business_policy.pdf", [
        ("Nova Retail Policy - Purchasing", ["Policy 1.1 Reorder trigger. A purchase review is required when stock falls to or below the reorder level.",
            "Policy 1.2 Target cover. Reorder quantities should restore 14 days of expected demand after the supplier lead time.",
            "Policy 1.3 Approval limits. Purchase orders up to USD 2,500 may be approved by the operations manager; larger orders need the owner.",
            "Policy 1.4 Safety buffer. Add a 3-day safety buffer to every supplier lead time when judging stockout risk."]),
        ("Nova Retail Policy - Returns and Service", ["Policy 2.1 Returns. Customers may return unused products within 30 days for a full refund.",
            "Policy 2.2 Service levels. Customer messages must receive a first reply within 2 business days.",
            "Policy 2.3 Complaint review. A product with three or more quality complaints in a month must be reviewed with the supplier.",
            "Policy 2.4 Actions. Reorders, supplier emails and customer follow-ups are drafts until a manager approves them."])])
    pdf("supplier_terms.pdf", [
        ("Supplier Terms - Nile Tech Distribution and Delta Home Supplies", [
            "Nile Tech Distribution. Standard lead time is 14 days from order confirmation. Minimum order quantity is 25 units per item. Payment terms are net 30.",
            "Nile Tech Distribution offers expedited air freight with a 7 day lead time for a surcharge of 12 percent of the order value.",
            "Delta Home Supplies. Standard lead time is 10 days (28 days for office chairs, which ship from overseas). Minimum order quantity is 10 units per item. Payment terms are net 45."]),
        ("Supplier Terms - BrightSource Wholesale and GreenLeaf Goods", [
            "BrightSource Wholesale. Standard lead time is 7 days. Minimum order quantity is 100 units for stationery and 20 units for accessories. Payment terms are net 15.",
            "GreenLeaf Goods. Standard lead time is 12 days. Minimum order quantity is 30 units per item. Payment terms are net 30.",
            "Defective goods must be reported within 14 days of delivery. Late deliveries beyond 5 days entitle Nova Retail to a 3 percent credit."])])

make_sample_data()
print(sorted(os.listdir(DATA)))

## Step 5 · Load the AI model (Qwen) once
**What it does:** downloads and loads **Qwen2.5-3B-Instruct**, a small open-source language model that runs inside Colab (no API, no key).

* The first run downloads ~6 GB, so it can take several minutes. Later runs reuse the cached copy.
* We load it **once** and every agent shares it (`llm_chat`). A lock makes sure two agents never use the model at the same moment.
* If loading fails (no GPU, out of memory, no internet) the notebook **keeps going** with plain templates — `llm_chat` just returns `None`.

**You should see:** `Model loaded on cuda` and a short test reply, or a warning that it continues without the model.

In [ ]:
llm_tok = llm_model = None
LLM_LOCK = threading.Lock()

if USE_LLM:
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM
        cuda = torch.cuda.is_available()
        if not cuda:
            print("WARNING: no GPU found. The model will be slow on CPU. Consider Runtime > Change runtime type > T4 GPU.")
        dt = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if cuda else torch.float32
        kw = {"device_map": "auto"} if cuda else {}
        if LOAD_IN_4BIT and cuda:
            from transformers import BitsAndBytesConfig
            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=dt)
        llm_tok = AutoTokenizer.from_pretrained(MODEL_ID); llm_tok.truncation_side = "left"   # if too long, drop the oldest text
        try:
            llm_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dt, **kw)
        except TypeError:                                                                     # older library versions
            llm_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dt, **kw)
        llm_model.eval()
        print("Model loaded on", llm_model.device)
    except Exception as e:
        llm_tok = llm_model = None
        print("Could not load the model:", str(e)[:200])
        print("Continuing WITHOUT the AI model (templates are used instead).")

def llm_chat(messages, max_new_tokens=300):
    """Ask the model. `messages` = [{"role": "system"|"user", "content": "..."}]. Returns text, or None if unavailable."""
    if llm_model is None:
        return None
    try:
        import torch
        text = llm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)   # Qwen's chat format
        inputs = llm_tok(text, return_tensors="pt", truncation=True, max_length=3072).to(llm_model.device)
        with LLM_LOCK, torch.inference_mode():
            out = llm_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=llm_tok.eos_token_id)
        return llm_tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    except Exception as e:                      # e.g. out of memory: never crash the whole workflow
        print("LLM call failed:", str(e)[:120]); return None

if llm_model is not None:
    print("Test reply:", llm_chat([{"role": "user", "content": "Reply with exactly: READY"}], 8))

## Step 6 · Small helper functions
**What they do:**

* `narrate(...)` – asks the AI to write a short summary **using only verified facts** we give it (answers are cached, and `USE_NARRATION=False` skips the AI). If there is no AI (or it fails) it returns our plain template text. We always keep the template (`fallback`) so the Verifier can swap it in if the AI made a mistake.
* `numbers_in / bad_numbers` – find numbers inside a text and check each one really exists in the facts. This is how we catch an AI that "invents" a number.
* `extract_json` – pulls a JSON object out of an AI reply (models sometimes add extra words around it).

**You should see:** no output (these are just definitions).

In [ ]:
NARRATE_CACHE = {}        # identical requests are answered instantly the second time

def narrate(role, facts, fallback, max_new_tokens=130):
    """AI-written summary of verified facts. Returns (text, source) where source is 'llm' or 'template'."""
    if not USE_NARRATION: return fallback, "template"
    key = hashlib.md5((role + json.dumps(facts, default=str)).encode()).hexdigest()
    if key in NARRATE_CACHE: return NARRATE_CACHE[key]
    prompt = ("Write a concise business summary (max 3 sentences) using ONLY the verified facts below. "
              "Do not add numbers, products or causes that are not in the facts. Mark guesses as hypotheses.\n\nFACTS:\n"
              + json.dumps(facts, default=str)[:3000])
    text = llm_chat([{"role": "system", "content": f"You are the {role} of a small retailer."}, {"role": "user", "content": prompt}], max_new_tokens)
    result = (text, "llm") if text and len(text) > 20 else (fallback, "template")
    NARRATE_CACHE[key] = result
    return result

def extract_json(text):
    m = re.search(r"\{.*\}", text or "", re.S)           # first '{' to last '}'
    return json.loads(m.group(0)) if m else None

def numbers_in(text):
    """Numbers worth checking in a text (ignores dates, IDs like P002, page refs, years and tiny counts like 'top 3')."""
    t = re.sub(r"\d{4}-\d{2}-\d{2}|\bP\d+\b|\bF\d+\b|\bp\.\d+\b|\b(?:19|20)\d\d\b|\b[A-Z]{2,4}-[0-9a-f]+\b", " ", text or "")
    out = []
    for m in re.finditer(r"\$?(\d[\d,]*\.?\d*)(%?)", t):
        v = float(m.group(1).replace(",", ""))
        if v <= 10 and v == int(v) and not m.group(0).startswith("$") and not m.group(2):
            continue
        out.append(v)
    return out

def all_numbers(obj, acc=None):
    """Collect every number that appears anywhere in a nested structure of facts."""
    acc = set() if acc is None else acc
    if isinstance(obj, bool) or obj is None: pass
    elif isinstance(obj, (int, float, np.integer, np.floating)): acc.update({float(obj), abs(float(obj))})   # a '-1.6%' change is written as 1.6 in text
    elif isinstance(obj, str): acc.update(numbers_in(obj))
    elif isinstance(obj, dict): [all_numbers(v, acc) for v in obj.values()]
    elif isinstance(obj, (list, tuple, set)): acc.add(float(len(obj))); [all_numbers(v, acc) for v in obj]
    return acc

def bad_numbers(text, facts):
    """Numbers in `text` that cannot be found in `facts` (these would be invented by the AI)."""
    allowed = all_numbers(facts) | {LOOKBACK_DAYS, SAFETY_DAYS, TARGET_COVER_DAYS, RISING_PCT, 100, 14}
    return [v for v in numbers_in(text) if not any(round(a) == v or abs(a - v) <= 0.051 or abs(a - v) <= 0.005 * abs(a) for a in allowed)]

## Agent 1 · Data Collector / Validator
**Job:** load the three data files, **check every row**, and separate good rows from bad ones.

For each file it looks for: missing columns, missing values, bad dates, negative or zero quantities/prices, ratings outside 1–5, and duplicates.
**Your files do not need our exact column names**: `canonicalize()` understands common alternatives (e.g. `Qty`→quantity, `SKU`→product_id, `Stars`→rating, `Vendor`→supplier), and `detect_kind()` works out whether an uploaded file is sales, inventory or feedback.

Bad rows are **not deleted silently** – they are moved to a "rejected" table together with the reason, so you can fix the source file.

* `validate(df, kind)` – checks one table. Returns `(clean_rows, rejected_rows, report)`.
* `reject_detail` – every rejected row gets a short code (`bad_rating`) **and** a plain-English reason (`rating '9' is above the maximum allowed rating of 5`). If your ratings use a 1–10 scale, set `RATING_MAX = 10` in this cell.
* `load_data(folder)` – reads all files (CSV or Excel), validates them, and stores everything in the global `DB` dictionary.
* `as_of` = the latest sale date. "Today" for all calculations is this date, so the demo data always makes sense.

**You should see:** a one-line quality summary per file (all rows valid for the sample data), the `as_of` date and the PDFs found.

In [ ]:
# ---------- settings for reading uploaded files ----------
RATING_MAX = 5          # highest rating your feedback file uses. Put 10 if your ratings are 1-10 (they are rescaled to 1-5 internally)

REQUIRED = {"sales": ["order_id", "date", "product_id", "product_name", "quantity", "unit_price"],
            "inventory": ["product_id", "product_name", "stock_quantity", "reorder_level", "unit_cost", "supplier", "lead_time_days"],
            "feedback": ["feedback_id", "date", "rating", "feedback_text", "product_id"]}
OPTIONAL = {"inventory": ["supplier"]}                          # may be missing from the file (filled with "Unknown supplier")
MIN_VALUE = {"sales": {"quantity": 1, "unit_price": 0.01}, "inventory": {"stock_quantity": 0, "reorder_level": 0, "unit_cost": 0.01, "lead_time_days": 0},
             "feedback": {"rating": 1}}
DUP_KEY = {"sales": None, "inventory": ["product_id"], "feedback": ["feedback_id"]}   # None = whole row

# Other names people commonly use for each column. Your file does NOT need our exact headers.
ALIASES = {
    "order_id": ["order", "orderid", "order_no", "order_number", "invoice", "invoice_id", "invoice_no", "invoice_number", "transaction_id", "transaction", "receipt_id", "receipt_no"],
    "date": ["order_date", "sale_date", "sales_date", "transaction_date", "invoice_date", "purchase_date", "created_at", "timestamp", "datetime", "day", "feedback_date", "review_date"],
    "product_id": ["productid", "product_code", "product_no", "product_number", "sku", "item_id", "item_code", "item_no", "article_id", "article_no", "code", "barcode"],
    "product_name": ["product", "productname", "item", "item_name", "name", "title", "product_title", "description"],
    "quantity": ["qty", "units", "units_sold", "quantity_sold", "qty_sold", "pcs", "pieces", "count"],
    "unit_price": ["price", "unitprice", "selling_price", "sale_price", "sales_price", "price_each", "price_per_unit", "unit_selling_price"],
    "stock_quantity": ["stock", "on_hand", "quantity_on_hand", "qty_on_hand", "in_stock", "stock_level", "current_stock", "stock_qty", "available", "inventory"],
    "reorder_level": ["reorder_point", "reorder", "reorder_qty_level", "min_stock", "minimum_stock", "reorder_threshold", "safety_stock", "min_level"],
    "unit_cost": ["cost", "cost_price", "purchase_price", "buy_price", "unit_cost_price", "cogs"],
    "supplier": ["vendor", "supplier_name", "vendor_name"],
    "lead_time_days": ["lead_time", "leadtime", "lead_days", "lead_time_day", "delivery_days", "supplier_lead_time", "lead_time_in_days"],
    "feedback_id": ["review_id", "comment_id", "feedback_no", "ticket_id", "survey_id"],
    "rating": ["stars", "star_rating", "score", "review_rating", "rate", "customer_rating"],
    "feedback_text": ["feedback", "comment", "comments", "review", "review_text", "text", "message", "customer_feedback", "remarks", "customer_comment"],
}
def _norm(c): return re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_")
_LOOKUP = {}
for _canon, _alts in ALIASES.items():
    _LOOKUP[_norm(_canon)] = _canon
    for _a in _alts: _LOOKUP.setdefault(_norm(_a), _canon)

def canonicalize(df):
    """Rename the columns of an uploaded table to the names the agents expect. Returns (table, {old_name: new_name} for renamed columns)."""
    ren, used, changed = {}, set(), {}
    for c in df.columns:                                         # exact names win over aliases
        n = _norm(c)
        if n in ALIASES and n not in used: ren[c] = n; used.add(n)
    for c in df.columns:
        if c in ren: continue
        canon = _LOOKUP.get(_norm(c))
        if canon and canon not in used: ren[c] = canon; used.add(canon); changed[str(c)] = canon
    out = df.rename(columns=ren)
    out.columns = [c if c in used else _norm(c) for c in out.columns]
    return out.loc[:, ~pd.Index(out.columns).duplicated()], changed

def fill_ids(df):
    """If a file has product names but no product ids (or the other way round), build the missing one so the files can still be linked."""
    df = df.copy()
    slug = lambda s: re.sub(r"[^A-Z0-9]+", "_", str(s).upper()).strip("_")
    if "product_id" not in df.columns and "product_name" in df.columns: df["product_id"] = df["product_name"].map(lambda s: slug(s) if pd.notna(s) else pd.NA)
    if "product_name" not in df.columns and "product_id" in df.columns: df["product_name"] = df["product_id"]
    return df

def detect_kind(df):
    """Guess whether a table is sales, inventory or feedback. Returns (kind, share_of_required_columns_found, missing_columns)."""
    d = fill_ids(canonicalize(df)[0])
    best, score, miss = None, -1, []
    for k, cols in REQUIRED.items():
        need = [c for c in cols if c not in OPTIONAL.get(k, [])]
        have = [c for c in need if c in d.columns]
        s = len(have) / len(need)
        if s > score: best, score, miss = k, s, [c for c in need if c not in d.columns]
    return best, score, miss

def parse_dates(s):
    """Read dates in almost any format. For 13/05/2026 style dates it works out whether the day or the month comes first."""
    s = s.astype("string")
    parts = s.str.extract(r"^\s*(\d{1,2})[/.\-](\d{1,2})[/.\-]\d{2,4}", expand=True).astype("Float64")
    dayfirst = bool((parts[0] > 12).fillna(False).any() and not (parts[1] > 12).fillna(False).any())
    return pd.to_datetime(s, errors="coerce", format="mixed", dayfirst=dayfirst).dt.normalize()

def validate(df, kind):
    """Check a table. Returns (clean, rejected, report). `rejected` has reject_reason (short codes) and reject_detail (plain-English explanation)."""
    df, renamed = canonicalize(df.copy()); df = fill_ids(df)
    for c in OPTIONAL.get(kind, []):
        if c not in df.columns: df[c] = pd.NA
    missing = [c for c in REQUIRED[kind] if c not in df.columns]
    if missing or len(df) == 0:
        return df.iloc[0:0], df, {"kind": kind, "rows": len(df), "clean": 0, "rejected": len(df), "usable": False, "renamed": renamed, "columns_seen": list(df.columns),
                                  "issues": {"missing_columns": missing} if missing else {"empty_file": 1}}
    d = df[REQUIRED[kind]].copy()
    for c in d.columns:                                           # trim spaces, treat "" as missing
        d[c] = d[c].astype("string").str.strip().replace("", pd.NA)
    why, detail = pd.Series("", index=d.index), pd.Series("", index=d.index)
    def flag(mask, reason, note=None):
        mask = mask.fillna(True).astype(bool)
        why[mask] += reason + ";"
        if note is not None: detail[mask] += note[mask] + "; "
    req = d.drop(columns=[c for c in OPTIONAL.get(kind, []) if c in d.columns])          # optional columns may be empty
    empty = req.isna()
    flag(empty.any(axis=1), "missing_value", "empty: " + empty.apply(lambda r: ", ".join(r.index[r.values.astype(bool)]), axis=1))
    raw = {c: d[c] for c in d.columns}
    if "date" in d:
        d["date"] = parse_dates(raw["date"])
        flag(raw["date"].notna() & d["date"].isna(), "bad_date", "date '" + raw["date"].fillna("") + "' is not a valid date")
    for col, low in MIN_VALUE[kind].items():                      # numbers: only flag values that are PRESENT but wrong (empty ones are 'missing_value')
        d[col] = pd.to_numeric(raw[col], errors="coerce")
        flag(raw[col].notna() & (d[col].isna() | (d[col] < low)), f"bad_{col}", f"{col} '" + raw[col].fillna("") + f"' must be a number >= {low:g}")
    if kind == "sales":
        flag(d["quantity"].notna() & (d["quantity"] % 1 != 0), "bad_quantity", "quantity '" + raw["quantity"].fillna("") + "' must be a whole number")
    if kind == "feedback":
        flag(d["rating"].notna() & (d["rating"] > RATING_MAX), "bad_rating", "rating '" + raw["rating"].fillna("") + f"' is above the maximum allowed rating of {RATING_MAX}")
    flag(d.duplicated(subset=DUP_KEY[kind], keep="first"), "duplicate", pd.Series("duplicate of an earlier row", index=d.index))
    ok = why == ""
    clean, rejected = d[ok].copy(), df[~ok].copy()
    rejected["reject_reason"] = why[~ok]; rejected["reject_detail"] = detail[~ok].str.rstrip("; ")
    if kind == "sales": clean["quantity"] = clean["quantity"].astype(int); clean["line_revenue"] = clean["quantity"] * clean["unit_price"]
    if kind == "inventory": clean["supplier"] = clean["supplier"].fillna("Unknown supplier")
    if kind == "feedback":
        clean["rating"] = clean["rating"].astype(float)
        if RATING_MAX != 5: clean["rating"] = (clean["rating"] * 5 / RATING_MAX).round(2)    # rescale e.g. 1-10 to 1-5
    issues = Counter(r for s in why[~ok] for r in s.split(";") if r)
    return clean.reset_index(drop=True), rejected, {"kind": kind, "rows": len(df), "clean": len(clean), "rejected": len(rejected), "issues": dict(issues), "usable": len(clean) > 0,
                                                    "renamed": renamed, "columns_seen": list(df.columns)}

def read_any(path):
    p = str(path).lower()
    if p.endswith((".xlsx", ".xls")): return pd.read_excel(path, dtype=str)
    try:                                                          # sniffs the separator (comma, semicolon, tab) and ignores a BOM
        return pd.read_csv(path, dtype=str, sep=None, engine="python", encoding="utf-8-sig", encoding_errors="replace")
    except Exception:
        return pd.read_csv(path, dtype=str, encoding="utf-8-sig", encoding_errors="replace")

DB = {}   # the loaded business data lives here: DB["sales"], DB["inventory"], DB["feedback"], DB["reports"], DB["as_of"], DB["pdfs"]

def load_data(folder=DATA):
    DB.update(sales=None, inventory=None, feedback=None, reports={}, rejected={}, warnings=[])
    names = {"sales": "sales", "inventory": "inventory", "feedback": "customer_feedback"}
    for kind, stem in names.items():
        path = next((f"{folder}/{stem}{e}" for e in (".csv", ".xlsx", ".xls") if os.path.exists(f"{folder}/{stem}{e}")), None)
        if path is None:
            DB["warnings"].append(f"{kind}: file not found (agents that need it will be skipped)"); continue
        try:
            DB[kind], DB["rejected"][kind], DB["reports"][kind] = validate(read_any(path), kind)
        except Exception as e:
            DB["warnings"].append(f"{kind}: could not read file ({e})")
    DB["as_of"] = DB["sales"]["date"].max() if DB["sales"] is not None and len(DB["sales"]) else None
    DB["pdfs"] = sorted(f"{folder}/{f}" for f in os.listdir(folder) if f.lower().endswith(".pdf"))
    for r in DB["reports"].values():
        if r["rejected"]: DB["warnings"].append(f"{r['kind']}: {r['rejected']} row(s) rejected {r['issues']}")
    ids = {k: set(DB[k]["product_id"].astype(str)) for k in ("sales", "inventory", "feedback") if DB[k] is not None and len(DB[k])}
    for k in ("sales", "feedback"):                              # files from different shops would give nonsense when mixed -> say so
        if "inventory" in ids and k in ids and ids[k] - ids["inventory"]:
            unk = sorted(ids[k] - ids["inventory"])
            DB["warnings"].append(f"{k}: {len(unk)} product_id(s) are not in the inventory file (e.g. {', '.join(unk[:3])}). Upload sales, inventory and feedback from the SAME shop together.")
    return DB

load_data()
for r in DB["reports"].values(): print(f"{r['kind']:10} {r['clean']}/{r['rows']} rows valid, {r['rejected']} rejected {r['issues']}")
print("as_of:", DB["as_of"].date(), "| PDFs:", [os.path.basename(p) for p in DB["pdfs"]])

## Agent 2 · Sales Analyst
**Job:** answer "how are sales doing?" with **exact numbers calculated by pandas**.

It computes, for a chosen period: revenue (`quantity × unit price`, summed), units sold, number of distinct orders, average order value (`revenue ÷ orders`), the top products, and the change versus the previous period of the same length.

* `parse_period(text)` turns phrases like *"last month"*, *"last 14 days"*, *"weekly"* into dates. "Last month" means the most recent **complete** calendar month in the data.
* If a number cannot be calculated (e.g. zero orders) it says so instead of guessing.
* The AI is only used at the end to write a short summary of those verified numbers.

**You should see:** a dictionary of September metrics and the summary sentence.

In [ ]:
def money(x): return "n/a" if x is None else f"{CURRENCY}{x:,.2f}"

def parse_period(text, as_of):
    """Return (start, end, label, was_explicit) for phrases like 'last month' or 'last 14 days'."""
    t, end = (text or "").lower(), as_of
    m = re.search(r"(?:last|past)\s+(\d{1,3})\s+days?", t)
    if m: n = int(m.group(1)); return end - pd.Timedelta(days=n - 1), end, f"last {n} days", True
    if re.search(r"week", t): return end - pd.Timedelta(days=6), end, "last 7 days", True
    if "this month" in t: return end.replace(day=1), end, end.strftime("%B %Y") + " (to date)", True
    if re.search(r"last month|monthly", t):
        month_end = (end + pd.offsets.MonthEnd(0)).normalize()
        if end == month_end: start, stop = end.replace(day=1), end                 # data ends on a month end -> that month is complete
        else: stop = end.replace(day=1) - pd.Timedelta(days=1); start = stop.replace(day=1)
        return start, stop, start.strftime("%B %Y"), True
    return end - pd.Timedelta(days=LOOKBACK_DAYS - 1), end, f"last {LOOKBACK_DAYS} days", False

def sales_metrics(df, start, end):
    cur = df[(df["date"] >= start) & (df["date"] <= end)]
    orders = cur["order_id"].nunique(); revenue = float(cur["line_revenue"].sum())
    return {"revenue": round(revenue, 2), "units": int(cur["quantity"].sum()), "orders": int(orders),
            "aov": round(revenue / orders, 2) if orders else None, "start": str(start.date()), "end": str(end.date())}

def analyze_sales(start=None, end=None, label=""):
    df = DB["sales"]
    if df is None or df.empty:
        return {"period": label, "notes": ["Insufficient data: no valid sales loaded."], "revenue": 0, "units": 0, "orders": 0, "aov": None,
                "top": [], "narrative": "No valid sales data is available.", "fallback": "No valid sales data is available.", "start": "", "end": ""}
    end = end if end is not None else DB["as_of"]; start = start if start is not None else end - pd.Timedelta(days=LOOKBACK_DAYS - 1)
    label = label or f"{start.date()} to {end.date()}"
    f = {"period": label, **sales_metrics(df, start, end), "notes": [], "change_pct": None}
    days = (end - start).days + 1
    if start - pd.Timedelta(days=days) >= df["date"].min():                       # is there a full previous period?
        prev = sales_metrics(df, start - pd.Timedelta(days=days), start - pd.Timedelta(days=1))
        f["previous_revenue"] = prev["revenue"]
        f["change_pct"] = round((f["revenue"] - prev["revenue"]) / prev["revenue"] * 100, 1) if prev["revenue"] else None
    else:
        f["notes"].append("Not enough history for a previous-period comparison.")
    if f["orders"] == 0: f["notes"].append("No sales in this period.")
    cur = df[(df["date"] >= start) & (df["date"] <= end)]
    g = cur.groupby("product_id").agg(name=("product_name", "first"), revenue=("line_revenue", "sum"), units=("quantity", "sum")).reset_index().sort_values("revenue", ascending=False)
    f["top"] = [{"product_id": r.product_id, "name": r.name, "revenue": round(float(r.revenue), 2), "units": int(r.units)} for r in g.head(5).itertuples()]
    fb = (f"{label}: revenue {money(f['revenue'])} from {f['units']} units in {f['orders']} orders (average order {money(f['aov'])}). "
          + (f"Top product: {f['top'][0]['name']} ({money(f['top'][0]['revenue'])}). " if f["top"] else "")
          + (f"Revenue changed {f['change_pct']:+.1f}% vs the previous period." if f["change_pct"] is not None else "No previous-period comparison available."))
    f["fallback"] = fb
    f["narrative"], f["narrative_source"] = narrate("Sales Analyst", {k: v for k, v in f.items() if k != "fallback"}, fb)
    return f

def rank_products(metric="revenue", order="highest", limit=5, start=None, end=None, product_ids=None):
    """Generic ranking tool: rank products by revenue, units or orders, highest or lowest first. Products with NO sales in the period count as 0."""
    df = DB["sales"]; end = end if end is not None else DB["as_of"]; start = start if start is not None else end - pd.Timedelta(days=LOOKBACK_DAYS - 1)
    cur = df[(df["date"] >= start) & (df["date"] <= end)]
    g = cur.groupby("product_id").agg(name=("product_name", "first"), revenue=("line_revenue", "sum"), units=("quantity", "sum"), orders=("order_id", "nunique")).reset_index()
    rows = {r.product_id: {"product_id": r.product_id, "name": r.name, "revenue": float(r.revenue), "units": int(r.units), "orders": int(r.orders)} for r in g.itertuples()}
    if DB.get("inventory") is not None:
        for c in DB["inventory"].itertuples(): rows.setdefault(c.product_id, {"product_id": c.product_id, "name": c.product_name, "revenue": 0.0, "units": 0, "orders": 0})
    out = [r for r in rows.values() if not product_ids or r["product_id"] in product_ids]
    out.sort(key=lambda r: (r[metric], r["revenue"], r["units"]), reverse=(order == "highest"))
    return [{**r, "revenue": round(r["revenue"], 2)} for r in out[:limit]]

s, e, lab, _ = parse_period("best sellers last month", DB["as_of"])
demo = analyze_sales(s, e, lab)
print({k: demo[k] for k in ("period", "revenue", "units", "orders", "aov", "change_pct")}); print(demo["narrative"])

## Agent 3 · Inventory Analyst
**Job:** find products that may **run out before the next delivery arrives**.

For every product it works out:
1. **Daily demand** = units sold in the last 30 days ÷ 30.
2. **Coverage** = stock ÷ daily demand (how many days the stock lasts).
3. It compares coverage to the supplier's **lead time**:

| Risk | Rule |
|---|---|
| 🔴 **critical** | out of stock while people still buy it, **or** coverage < lead time |
| 🟠 **high** | coverage < lead time + 3 safety days |
| 🟡 **medium** | stock is at/below the reorder level (but coverage looks OK) |
| 🟢 **low** | none of the above |

It also suggests a **draft reorder quantity** = `demand × (lead time + 14 days) − stock`, and a **demand growth %** (last 14 days vs the 14 before) that the Coordinator uses.
These are *assumptions* (demand stays the same, no seasonality) and are listed in the result. It never promises a stockout date it cannot support.

**You should see:** the critical/high/medium counts and a small table of the riskiest products.

In [ ]:
RISK_ORDER = {"critical": 0, "high": 1, "medium": 2, "unknown": 3, "low": 4}

def classify_risk(stock, reorder, lead, demand):
    """Return (risk, coverage_days). Pure rule-based, no AI."""
    below = stock <= reorder
    if demand is None: return ("medium" if below else "unknown"), None
    if demand <= 0:    return ("medium" if below else "low"), None
    cov = stock / demand
    if stock <= 0 or cov < lead: return "critical", cov
    if cov < lead + SAFETY_DAYS: return "high", cov
    return ("medium" if below else "low"), cov

def analyze_inventory():
    inv, sales, as_of = DB["inventory"], DB["sales"], DB["as_of"]
    if inv is None or inv.empty:
        return {"items": [], "counts": {}, "assumptions": [], "narrative": "No valid inventory data.", "fallback": "No valid inventory data.", "notes": ["Insufficient data."]}
    demand, growth = {}, {}
    if sales is not None and len(sales):
        win = sales[sales["date"] > as_of - pd.Timedelta(days=LOOKBACK_DAYS)]
        units = win.groupby("product_id")["quantity"].sum()
        for pid in inv["product_id"]: demand[pid] = float(units.get(pid, 0)) / LOOKBACK_DAYS
        rec = sales[sales["date"] > as_of - pd.Timedelta(days=14)].groupby("product_id")["quantity"].sum()
        pri = sales[(sales["date"] <= as_of - pd.Timedelta(days=14)) & (sales["date"] > as_of - pd.Timedelta(days=28))].groupby("product_id")["quantity"].sum()
        for pid in inv["product_id"]:
            if pri.get(pid, 0) >= 10: growth[pid] = round((rec.get(pid, 0) - pri[pid]) / pri[pid] * 100, 1)   # need enough data to trust a %
    items = []
    for r in inv.itertuples():
        d = demand.get(r.product_id); risk, cov = classify_risk(r.stock_quantity, r.reorder_level, r.lead_time_days, d)
        qty = None
        if d and d > 0 and risk != "low":
            qty = max(0, math.ceil(d * (r.lead_time_days + TARGET_COVER_DAYS) - r.stock_quantity))
        items.append({"product_id": r.product_id, "name": r.product_name, "supplier": r.supplier, "stock": r.stock_quantity, "reorder_level": r.reorder_level,
                      "lead_time": r.lead_time_days, "daily_demand": None if d is None else round(d, 3), "coverage": None if cov is None else round(cov, 1),
                      "growth_pct": growth.get(r.product_id), "below_reorder": r.stock_quantity <= r.reorder_level, "risk": risk,
                      "reorder_qty": qty, "est_cost": round(qty * r.unit_cost, 2) if qty else None})
    items.sort(key=lambda i: (RISK_ORDER[i["risk"]], i["coverage"] if i["coverage"] is not None else 1e9))
    counts = Counter(i["risk"] for i in items)
    attention = [i for i in items if i["risk"] in ("critical", "high", "medium")]
    levels = [(lvl, [i["name"] for i in items if i["risk"] == lvl]) for lvl in ("critical", "high", "medium")]
    if attention:                                              # every product is named under its TRUE risk level
        fb = (f"{len(attention)} of {len(items)} products need attention: " + "; ".join(f"{len(n)} {lvl} ({', '.join(n)})" for lvl, n in levels if n)
              + ". Reorder quantities are drafts based on the assumptions below.")
    else: fb = f"No product needs attention right now ({len(items)} checked)."
    f = {"as_of": str(as_of.date()) if as_of is not None else "", "items": items, "counts": dict(counts), "notes": [],
         "assumptions": [f"Daily demand = units sold in the last {LOOKBACK_DAYS} days / {LOOKBACK_DAYS}; assumed to continue unchanged.",
                         f"Safety buffer = {SAFETY_DAYS} days on top of supplier lead time.",
                         f"Draft reorder qty = demand x (lead time + {TARGET_COVER_DAYS} days) - stock; needs human review."], "fallback": fb}
    f["narrative"], f["narrative_source"] = fb, "template"      # written from the data, NOT by the AI (a small AI model mixed up risk levels here)
    return f

inv_demo = analyze_inventory()
print(inv_demo["counts"])
print(pd.DataFrame(inv_demo["items"])[["product_id", "name", "risk", "stock", "coverage", "lead_time", "growth_pct", "reorder_qty"]].head(6).to_string(index=False))

## Agent 4 · Customer Feedback Analyst
**Job:** sort customer comments into themes — **delivery, quality, price, availability, customer_service, other** — and report what is repeated.

How it works (two layers, so it never breaks):
1. **Keyword classifier** (always available): looks for words such as *late, broke, expensive, sold out, refund*.
2. **The AI (Qwen)**, if loaded and `FEEDBACK_USE_LLM=True`: classifies comments in small batches (each comment is classified **once**, then cached) and answers in JSON. Each answer is **validated with Pydantic** (a library that checks data shapes). Invalid or made-up items are thrown away and the keyword classifier is used for them instead.

Counting, averages and the list of feedback IDs are done by Python. The result keeps **facts** (counts + IDs) separate from **hypotheses** (guesses, always labelled "Hypothesis").

**You should see:** the method used (`keyword` or `llm`/`mixed`), the average rating and a theme table.

In [ ]:
from pydantic import BaseModel, ValidationError, Field
THEMES = ["delivery", "quality", "price", "availability", "customer_service", "other"]

class FBItem(BaseModel):                              # the exact shape we accept from the AI
    feedback_id: str
    themes: list[Literal["delivery", "quality", "price", "availability", "customer_service", "other"]] = Field(min_length=1)   # an empty list is rejected
    sentiment: Literal["positive", "neutral", "negative"]

LEXICON = {"delivery": r"deliver\w*|shipping|courier|parcel|package|tracking|arrived late|delayed",
           "quality": r"broke\w*|defect\w*|flimsy|faulty|flicker\w*|stopped working|build quality|poor quality|fell apart|scratch\w*",
           "price": r"price\w*|expensive|overpriced|pricey|discount",
           "availability": r"out of stock|sold out|unavailable|restock\w*|back[- ]order\w*|in stock",
           "customer_service": r"support|customer service|staff|rude|refund\w*|unhelpful|ignored"}
POSITIVE = r"great|excellent|love|perfect|happy|recommend|solid"

def keyword_classify(text, rating):
    t = (text or "").lower()
    themes = [k for k, rx in LEXICON.items() if re.search(rf"\b(?:{rx})\b", t)]
    if themes and re.search(rf"\b(?:{POSITIVE})\b", t) and (rating is None or rating >= 4): themes = []   # praise is not a complaint
    sentiment = "negative" if rating is not None and rating <= 2 else "positive" if rating is not None and rating >= 4 else "neutral"
    return {"themes": themes or ["other"], "sentiment": sentiment}

CLS_CACHE = {}   # (feedback_id, text) -> (classification, from_ai). Each comment is sent to the AI only once

def classify_with_llm(rows):
    """Ask Qwen for themes of up to 8 comments. Returns {feedback_id: {...}} for the items that passed validation."""
    lines = "\n".join(f'{fid}: "{txt[:250]}"' for fid, txt in rows)
    prompt = (f"Classify each customer comment. Allowed themes: {', '.join(THEMES)} (a comment may have several; use 'other' for general praise). "
              'Return ONLY JSON like {"items":[{"feedback_id":"F001","themes":["delivery"],"sentiment":"negative"}]} with one item per id.\n\n' + lines)
    try:
        data = extract_json(llm_chat([{"role": "user", "content": prompt}], max_new_tokens=40 * len(rows) + 40))
        out, wanted = {}, {r[0] for r in rows}
        for raw in (data or {}).get("items", []):
            try:
                item = FBItem.model_validate(raw)
                if item.feedback_id in wanted: out[item.feedback_id] = {"themes": item.themes or ["other"], "sentiment": item.sentiment}   # ignore invented ids
            except ValidationError: pass                                                                             # skip bad items
        return out
    except Exception:
        return {}

def analyze_feedback(start=None, end=None):
    fb = DB["feedback"]
    if fb is None or fb.empty:
        return {"total": 0, "themes": [], "issues": [], "hypotheses": [], "narrative": "No valid feedback data.", "fallback": "No valid feedback data.", "avg_rating": None, "method": "none", "notes": ["Insufficient data."]}
    if start is not None: fb = fb[(fb["date"] >= start) & (fb["date"] <= end)]
    cls, ai_count = {}, 0
    rows = list(zip(fb["feedback_id"], fb["feedback_text"]))
    for fid, txt in rows:                                               # reuse earlier answers (CLS_CACHE stores: result, was it from the AI?)
        if (fid, txt) in CLS_CACHE:
            cls[fid], was_ai = CLS_CACHE[(fid, txt)]; ai_count += was_ai
    ai_on = llm_model is not None and FEEDBACK_USE_LLM
    if ai_on:
        todo = [(f, t) for f, t in rows if f not in cls]
        for i in range(0, len(todo), 10):
            print(f"  feedback AI: {min(i + 10, len(todo))}/{len(todo)} comments", end="\r")      # progress
            batch = todo[i:i + 10]; got = classify_with_llm(batch)
            for f, t in batch:
                if f in got: cls[f] = got[f]; CLS_CACHE[(f, t)] = (got[f], True); ai_count += 1
    for r in fb.itertuples():                                         # keyword fallback for anything the AI did not cover
        if r.feedback_id not in cls:
            cls[r.feedback_id] = keyword_classify(r.feedback_text, r.rating)
            if ai_on: CLS_CACHE[(r.feedback_id, r.feedback_text)] = (cls[r.feedback_id], False)   # remember it, so the AI is not asked again
    method = "keyword" if ai_count == 0 else "llm" if ai_count == len(fb) else "mixed"

    meta = fb.set_index("feedback_id")
    by_theme, by_pt = {}, {}
    for fid, c in cls.items():
        for th in c["themes"]:
            by_theme.setdefault(th, []).append((fid, c["sentiment"]))
            if th != "other" and c["sentiment"] != "positive": by_pt.setdefault((meta.loc[fid, "product_id"], th), []).append(fid)
    themes = sorted([{"theme": th, "count": len(v), "negative": sum(s == "negative" for _, s in v), "ids": [i for i, _ in v]} for th, v in by_theme.items()],
                    key=lambda x: (-x["negative"], -x["count"]))
    issues = [{"product_id": p, "theme": th, "count": len(ids), "ids": ids, "avg_rating": round(float(meta.loc[ids, "rating"].mean()), 2)}
              for (p, th), ids in sorted(by_pt.items(), key=lambda kv: -len(kv[1])) if len(ids) >= 3]
    avg = round(float(fb["rating"].mean()), 2)
    hyp_text = {"delivery": "delivery complaints may come from carrier or supplier delays", "quality": "repeated quality complaints may point to a defective batch",
                "price": "price complaints may reflect competitor pricing", "availability": "availability complaints may be linked to low stock or long lead times",
                "customer_service": "service complaints may reflect slow replies"}
    hyps = [f"Hypothesis: {hyp_text[t['theme']]} (not verified)." for t in themes if t["theme"] in hyp_text and t["negative"]][:3]
    obs = [f"{len(fb)} feedback items, average rating {avg}."] + [f"Theme '{t['theme']}': {t['negative']} negative of {t['count']} mentions (e.g. {', '.join(t['ids'][:3])})." for t in themes if t["theme"] != "other" and t["negative"]][:3]
    fallback = " ".join(obs[:3] + hyps[:1])
    f = {"total": len(fb), "avg_rating": avg, "themes": themes, "issues": issues, "observations": obs, "hypotheses": hyps, "method": method, "notes": [], "fallback": fallback}
    f["narrative"], f["narrative_source"] = narrate("Customer Feedback Analyst", {k: v for k, v in f.items() if k != "fallback"}, fallback)
    return f

fb_demo = analyze_feedback()
print("method:", fb_demo["method"], "| average rating:", fb_demo["avg_rating"])
print(pd.DataFrame(fb_demo["themes"])[["theme", "count", "negative"]].to_string(index=False)); print(fb_demo["hypotheses"][:2])

## Agent 5 · Business Document Researcher (RAG)
**Job:** answer questions from the **PDF documents** and always say **where** the answer came from.

Steps (this is "RAG"):
1. **Read** every PDF page with `pypdf` and keep the file name + page number.
2. **Split** pages into small chunks (LangChain's text splitter).
3. **Embed** chunks: turn each into a list of numbers with a Sentence-Transformers model (`all-MiniLM-L6-v2`). If that library is unavailable the notebook uses a simple word-hashing fallback and tells you.
4. **Search** with **FAISS**: find the chunks most similar to your question.
5. **Answer** with Qwen using *only* those chunks, citing `[file.pdf p.N]`. If nothing relevant is found it says **INSUFFICIENT EVIDENCE** instead of guessing.

Safety checks: citations the AI makes up (not in the retrieved chunks) are removed, and if the AI gives no citation we add the top source.

**You should see:** how many chunks were indexed, then a sample answer with its citation.

In [ ]:
import faiss
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate

try:                                                                   # best: a real semantic embedding model
    from sentence_transformers import SentenceTransformer
    _st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embed = lambda texts: np.asarray(_st.encode(list(texts), normalize_embeddings=True), dtype="float32")
    EMBEDDER, MIN_SCORE, LEXICAL_GATE = "MiniLM (sentence-transformers)", 0.30, False
except Exception as e:                                                 # fallback: hashed bag-of-words (works anywhere, less smart)
    print("sentence-transformers not available -> using simple word-hashing embeddings.", str(e)[:60])
    def embed(texts):
        out = np.zeros((len(texts), 512), dtype="float32")
        for i, t in enumerate(texts):
            for w in re.findall(r"[a-z0-9]+", t.lower()): out[i, int(hashlib.md5(w.encode()).hexdigest(), 16) % 512] += 1
        return out / np.maximum(np.linalg.norm(out, axis=1, keepdims=True), 1e-9)
    EMBEDDER, MIN_SCORE, LEXICAL_GATE = "word-hashing fallback", 0.10, True

INDEX = {"chunks": [], "faiss": None}

def build_index(pdf_paths):
    chunks, splitter = [], RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
    for path in pdf_paths:
        for page_no, page in enumerate(PdfReader(path).pages, start=1):
            text = (page.extract_text() or "").strip()
            for piece in splitter.split_text(text):
                chunks.append({"source": os.path.basename(path), "page": page_no, "text": piece})   # keep file + page for citations
    INDEX["chunks"] = chunks
    INDEX["faiss"] = None
    if chunks:
        vecs = embed([c["text"] for c in chunks])
        INDEX["faiss"] = faiss.IndexFlatIP(vecs.shape[1]); INDEX["faiss"].add(vecs)     # inner product of unit vectors = cosine similarity
    return len(chunks)

def search_docs(question, k=4):
    if INDEX["faiss"] is None: return []
    scores, ids = INDEX["faiss"].search(embed([question]), min(k, len(INDEX["chunks"])))
    hits = [{**INDEX["chunks"][i], "score": float(s)} for s, i in zip(scores[0], ids[0]) if i >= 0 and s >= MIN_SCORE]
    if LEXICAL_GATE:                                       # the hashing fallback needs a word-overlap sanity check
        q = {w[:5] for w in re.findall(r"[a-z0-9]+", question.lower()) if len(w) > 3}
        hits = [h for h in hits if q & {w[:5] for w in re.findall(r"[a-z0-9]+", h["text"].lower())}]
    return hits

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the numbered context. Cite every claim like [file.pdf p.2]. If the context does not answer, reply exactly: INSUFFICIENT EVIDENCE. Never invent terms, prices or policies."),
    ("human", "Context:\n{context}\n\nQuestion: {question}")])

def quote_answer(hits):
    """No-AI answer: quote the two best passages with their citations."""
    top = hits[:2]
    return {"answer": "Relevant excerpts:\n" + "\n".join(f"- \"{' '.join(h['text'].split())[:300]}\" [{h['source']} p.{h['page']}]" for h in top),
            "citations": [(h["source"], h["page"]) for h in top], "hits": hits, "sufficient": True}

def ask_docs(question):
    hits = search_docs(question)
    if not INDEX["chunks"]: return {"answer": "No documents are indexed.", "citations": [], "hits": [], "sufficient": False}
    if not hits: return {"answer": "INSUFFICIENT EVIDENCE: the documents do not clearly answer this. Nothing was assumed.", "citations": [], "hits": [], "sufficient": False}
    allowed = {(h["source"], h["page"]) for h in hits}
    context = "\n\n".join(f"[{i+1}] ({h['source']} p.{h['page']})\n{h['text']}" for i, h in enumerate(hits))
    msgs = [{"role": "system" if m.type == "system" else "user", "content": m.content} for m in RAG_PROMPT.format_messages(context=context, question=question)]
    text = llm_chat(msgs, 300)
    if text is None: return quote_answer(hits)             # no AI: quote the best passages instead
    if "INSUFFICIENT EVIDENCE" in text.upper(): return {"answer": "INSUFFICIENT EVIDENCE: the retrieved passages do not answer the question.", "citations": [], "hits": hits, "sufficient": False}
    cited = [(m.group(1).strip(), int(m.group(2))) for m in re.finditer(r"\[([^\[\]]+?)\s+p\.(\d+)\]", text)]
    for src, pg in [c for c in cited if c not in allowed]: text = text.replace(f"[{src} p.{pg}]", "")        # remove invented citations
    valid = [c for c in cited if c in allowed] or [(hits[0]["source"], hits[0]["page"])]
    if not [c for c in cited if c in allowed]: text += f"\n\nSource (added by QA): [{valid[0][0]} p.{valid[0][1]}]"
    return {"answer": text.strip(), "citations": list(dict.fromkeys(valid)), "hits": hits, "sufficient": True}

print("indexed chunks:", build_index(DB["pdfs"]), "| embedder:", EMBEDDER)
r = ask_docs("What is the standard lead time for Nile Tech Distribution?"); print(r["answer"][:300], "\ncitations:", r["citations"])
print(ask_docs("Who won the football world cup?")["answer"])

## Agent 6 · Operations Coordinator
**Job:** look **across** the specialists' results and spot problems no single agent can see.

Example rule: *rising demand + low stock + long supplier lead time → stockout risk*. For every risky product it collects **evidence** from several sources and gives a **priority score**:

`score = urgency × 1.5 + impact + evidence × 0.5`

| Part | How it is measured |
|---|---|
| **urgency** | critical = 3, high = 2, medium = 1 (+0.5 if demand is rising) |
| **impact** | how much of the period's revenue the product makes (big share = 3) |
| **evidence** | how many independent sources support it (inventory, sales, feedback IDs, PDF page) – max 3 |

Score ≥ 8 → **high**, ≥ 5.5 → **medium**, otherwise **low**. Each action gets an ID, a title, a description, evidence references (like `inventory:P002`, `feedback:F015`, `doc:supplier_terms.pdf p.1`), a rationale and a suggested next step. Every new action starts with status `proposed`. The Coordinator **never invents** anything: it only combines what the other agents found.

**You should see:** the top actions with their priority and evidence.

In [ ]:
def label(score): return "high" if score >= 8 else "medium" if score >= 5.5 else "low"

def coordinate(sales, inv, fb, doc_hits):
    actions = []
    total = (sales or {}).get("revenue") or 0
    share = {t["product_id"]: t["revenue"] / total for t in (sales or {}).get("top", [])} if total else {}
    fb_issues = {}
    for iss in (fb or {}).get("issues", []): fb_issues.setdefault(iss["product_id"], []).append(iss)
    for it in (inv or {}).get("items", []):
        if it["risk"] not in ("critical", "high", "medium"): continue
        urgency = {"critical": 3.0, "high": 2.0, "medium": 1.0}[it["risk"]]
        ev, parts = [f"inventory:{it['product_id']}"], [f"{it['name']}: stock {it['stock']:g}"]
        if it["coverage"] is not None: parts.append(f"{it['coverage']} days of stock vs {it['lead_time']:g}-day supplier lead time")
        if it["growth_pct"] is not None and it["growth_pct"] >= RISING_PCT:
            parts.append(f"demand up {it['growth_pct']:.0f}% over the last 14 days vs the 14 before"); ev.append(f"sales:{it['product_id']}"); urgency += 0.5
        elif it["product_id"] in share: ev.append(f"sales:{it['product_id']}")
        complaints = [x for x in fb_issues.get(it["product_id"], []) if x["theme"] in ("availability", "delivery")]
        if complaints:
            parts.append(f"{sum(x['count'] for x in complaints)} customer complaints about availability/delivery"); ev += [f"feedback:{i}" for i in complaints[0]["ids"][:3]]
        for h in doc_hits or []:                                          # supplier terms from the PDFs, if they mention this supplier
            if it["supplier"].split()[0].lower() in h["text"].lower(): ev.append(f"doc:{h['source']} p.{h['page']}"); break
        sh = share.get(it["product_id"], 0); impact = 3.0 if sh >= 0.10 else 2.0 if sh >= 0.04 else 1.0
        score = round(urgency * 1.5 + impact + min(3, len(ev)) * 0.5, 2)
        title = (f"Urgent: possible stockout of {it['name']} ({it['product_id']})" if it["risk"] == "critical" or (it["risk"] == "high" and urgency > 2)
                 else f"Review reorder for {it['name']} ({it['product_id']})")
        qty = f" Suggested draft quantity: {it['reorder_qty']} units (needs review)." if it["reorder_qty"] else ""
        actions.append({"id": "ACT-" + uuid.uuid4().hex[:8], "title": title, "description": "; ".join(parts) + ".", "priority": label(score), "score": score,
                        "evidence": ev, "product_id": it["product_id"], "status": "proposed",
                        "rationale": f"Urgency {urgency:g}, impact {impact:g} ({sh*100:.0f}% of revenue), evidence {min(3, len(ev))} source(s).",
                        "next_step": f"Confirm lead time with {it['supplier']} and approve or edit the draft purchase review." + qty})
    for pid, lst in fb_issues.items():                                    # policy: 3+ quality complaints -> review with the supplier
        for iss in lst:
            if iss["theme"] == "quality":
                ev = [f"feedback:{i}" for i in iss["ids"][:4]]; score = round(3.0 + 1.0 + min(3, len(ev)) * 0.5, 2)
                actions.append({"id": "ACT-" + uuid.uuid4().hex[:8], "title": f"Investigate recurring quality complaints for product {pid}", "description": f"{iss['count']} quality complaints, average rating {iss['avg_rating']}.",
                                "priority": label(score), "score": score, "evidence": ev, "product_id": pid, "status": "proposed",
                                "rationale": "Observed repeated complaints; the cause is a hypothesis until the product is inspected.",
                                "next_step": "Inspect stock for defects and review with the supplier (see defect terms in the supplier PDF)."})
    actions.sort(key=lambda a: -a["score"])
    top = actions[:6]
    fallback = ("Top priorities: " + " ".join(f"{i+1}) {a['title']} [{a['priority']}] - {a['description']}" for i, a in enumerate(top[:3]))) if top else "No urgent risks were found in the available data."
    summary, src = narrate("Operations Coordinator", {"actions": [{k: a[k] for k in ("title", "priority", "description")} for a in top[:3]]}, fallback)
    return {"actions": top, "summary": summary, "summary_fallback": fallback, "summary_source": src}

co_demo = coordinate(demo, inv_demo, fb_demo, search_docs("supplier lead time minimum order"))
for a in co_demo["actions"][:4]: print(f"[{a['priority']:6}] {a['score']:5}  {a['title']}\n          evidence: {', '.join(a['evidence'])}")

## Agent 7 · Verification / QA
**Job:** **never trust the AI blindly.** Before any answer is shown, the Verifier re-checks everything it can:

| Check | How |
|---|---|
| Sales numbers | recomputes revenue, units and orders with a **separate plain-Python loop** and compares |
| Inventory risk | recomputes daily demand and the risk level independently and compares |
| Feedback | every cited feedback ID exists, and each theme's count equals its list of IDs |
| Document answers | every citation must be one of the retrieved passages |
| Actions | evidence references resolve to real products / feedback IDs / PDF pages, and each action has a rationale and next step |
| Written summaries | every number in a summary must exist in the verified facts (catches invented numbers) |

If a check fails, the **correction step** swaps the AI text for the safe template text (or removes bad evidence) and verifies again – once. If it still fails, the answer is shown with a clear warning for human review.

**You should see:** `PASSED` with the number of checks run.

In [ ]:
def evidence_ok(ev):
    """Does an evidence reference like 'inventory:P002', 'feedback:F015' or 'doc:file.pdf p.1' point to something real?"""
    kind, _, ref = ev.partition(":")
    pids = set(DB["inventory"]["product_id"]) if DB["inventory"] is not None else set()
    fids = set(DB["feedback"]["feedback_id"]) if DB["feedback"] is not None else set()
    if kind in ("inventory", "sales"): return ref in pids
    if kind == "feedback": return ref in fids
    if kind == "doc" and " p." in ref:
        src, page = ref.rsplit(" p.", 1)
        return any(c["source"] == src and str(c["page"]) == page for c in INDEX["chunks"])
    return False

def verify(sales=None, inv=None, fb=None, rag=None, actions=None, summary=None):
    errors, count = [], [0]
    def check(ok, msg):
        count[0] += 1
        if not ok: errors.append(msg)
    clean = lambda d: {k: v for k, v in d.items() if k not in ("narrative", "fallback")}
    if sales and sales.get("start") and DB["sales"] is not None:
        rev = units = 0; orders = set()
        for r in DB["sales"].itertuples():                                  # independent recomputation (plain Python loop)
            if sales["start"] <= str(r.date.date()) <= sales["end"]: rev += r.quantity * r.unit_price; units += r.quantity; orders.add(r.order_id)
        check(abs(rev - sales["revenue"]) < 0.02, f"sales revenue {sales['revenue']} != recomputed {rev:.2f}")
        check(units == sales["units"], f"sales units {sales['units']} != {units}")
        check(len(orders) == sales["orders"], f"sales orders {sales['orders']} != {len(orders)}")
        check(not bad_numbers(sales["narrative"], clean(sales)), "sales summary contains numbers that are not in the data")
        if sales.get("rank"):                                               # a ranking is recomputed independently (plain loop) and its ORDER is checked
            per = {}
            for r in DB["sales"].itertuples():
                if sales["start"] <= str(r.date.date()) <= sales["end"]:
                    p = per.setdefault(r.product_id, [0.0, 0, set()]); p[0] += r.quantity * r.unit_price; p[1] += r.quantity; p[2].add(r.order_id)
            for t in sales["rank"]:
                rv, un, od = per.get(t["product_id"], [0.0, 0, set()])
                check(abs(rv - t["revenue"]) < 0.02 and un == t["units"] and len(od) == t["orders"], f"ranking row {t['product_id']} does not match the recomputed totals")
            mt = sales["rank_meta"]; vals = [t[mt["metric"]] for t in sales["rank"]]
            check(vals == sorted(vals, reverse=(mt["order"] == "highest")), "ranking is not in the requested order")
    if inv and inv.get("items") and DB["sales"] is not None:
        win = DB["sales"][DB["sales"]["date"] > DB["as_of"] - pd.Timedelta(days=LOOKBACK_DAYS)].groupby("product_id")["quantity"].sum()
        for it in inv["items"]:
            d = float(win.get(it["product_id"], 0)) / LOOKBACK_DAYS
            check(it["daily_demand"] is not None and abs(d - it["daily_demand"]) < 0.01, f"{it['product_id']}: demand mismatch")
            cov = it["stock"] / d if d > 0 else None
            exp = ("critical" if (it["stock"] <= 0 and d > 0) or (cov is not None and cov < it["lead_time"]) else
                   "high" if cov is not None and cov < it["lead_time"] + SAFETY_DAYS else "medium" if it["stock"] <= it["reorder_level"] else "low")
            check(exp == it["risk"], f"{it['product_id']}: risk '{it['risk']}' but expected '{exp}'")
        check(not bad_numbers(inv["narrative"], clean(inv)), "inventory summary contains numbers that are not in the data")
    if fb and fb.get("themes") and DB["feedback"] is not None:
        known = set(DB["feedback"]["feedback_id"])
        for t in fb["themes"]:
            check(t["count"] == len(t["ids"]), f"feedback theme {t['theme']}: count != number of ids")
            check(all(i in known for i in t["ids"]), f"feedback theme {t['theme']}: unknown feedback id")
        check(not bad_numbers(fb["narrative"], clean(fb)), "feedback summary contains numbers that are not in the data")
    if rag:
        got = {(h["source"], h["page"]) for h in rag["hits"]}
        check(all(tuple(c) in got for c in rag["citations"]), "document answer cites a passage that was not retrieved")
        if rag["sufficient"]: check(bool(rag["citations"]), "document answer has no citation")
    if actions:
        for a in actions:
            check(bool(a["evidence"]) and bool(a["rationale"]) and bool(a["next_step"]), f"{a['id']}: missing evidence, rationale or next step")
            check(a["status"] == "proposed", f"{a['id']}: new actions must start as 'proposed'")
            for ev in a["evidence"]:
                check(evidence_ok(ev), f"{a['id']}: evidence '{ev}' cannot be found")
    if summary:
        check(not bad_numbers(summary, [actions, inv and clean(inv)]), "coordinator summary contains numbers that are not in the data")
    return {"passed": not errors, "checks": count[0], "errors": errors}

v = verify(demo, inv_demo, fb_demo, ask_docs("lead time for Nile Tech"), co_demo["actions"], co_demo["summary"])
print("PASSED" if v["passed"] else "FAILED", "-", v["checks"], "checks", v["errors"][:3])

## Agent 8 · Reporter / Action Manager (SQLite)
**Job:** turn findings into things a person can use, and remember them.

* Saves **proposed actions** in a small SQLite database (`nova.sqlite`) with an ID, priority, evidence, rationale, next step and **status**.
* Status can only move along safe paths: `proposed → approved | rejected` and `approved → completed | rejected`. Every change is stored in a history table with a timestamp.
* Writes a **weekly report** (Markdown) and CSV files (actions + inventory review list).
* Creates **drafts** of supplier and customer messages. They are labelled **DRAFT – NOT SENT**: this program never sends an email or places an order.
* All SQL uses `?` placeholders (safe against SQL injection).

**You should see:** `Database ready`.

In [ ]:
import tempfile
def conn():
    c = sqlite3.connect(DB_PATH); c.row_factory = sqlite3.Row; return c

def init_db():
    with conn() as c:
        c.executescript("""
        CREATE TABLE IF NOT EXISTS actions(id TEXT PRIMARY KEY, run_id TEXT, created TEXT, title TEXT, description TEXT, priority TEXT, score REAL,
                                           evidence TEXT, rationale TEXT, next_step TEXT, status TEXT, product_id TEXT);
        CREATE TABLE IF NOT EXISTS history(id INTEGER PRIMARY KEY AUTOINCREMENT, action_id TEXT, ts TEXT, from_status TEXT, to_status TEXT, note TEXT);
        CREATE TABLE IF NOT EXISTS runs(id TEXT PRIMARY KEY, ts TEXT, request TEXT, intent TEXT, status TEXT, seconds REAL, steps TEXT, verification TEXT);
        CREATE TABLE IF NOT EXISTS drafts(id TEXT PRIMARY KEY, run_id TEXT, kind TEXT, recipient TEXT, subject TEXT, body TEXT, status TEXT);
        CREATE TABLE IF NOT EXISTS reports(id TEXT PRIMARY KEY, ts TEXT, run_id TEXT, title TEXT, md_path TEXT, csv_paths TEXT);""")
now = lambda: time.strftime("%Y-%m-%d %H:%M:%S")
TRANSITIONS = {"proposed": {"approved", "rejected"}, "approved": {"completed", "rejected"}, "rejected": set(), "completed": set()}

def save_actions(run_id, actions):
    with conn() as c:
        for a in actions:
            old = c.execute("SELECT id, status FROM actions WHERE title=? AND status IN ('proposed','approved')", (a["title"],)).fetchone()
            if old:                                                # same proposal already open: keep its id and status...
                a["id"] = old["id"]
                if old["status"] == "proposed":                    # ...but refresh its numbers, because the data may have changed since it was first made
                    c.execute("UPDATE actions SET run_id=?, description=?, priority=?, score=?, evidence=?, rationale=?, next_step=?, product_id=? WHERE id=?",
                              (run_id, a["description"], a["priority"], a["score"], json.dumps(a["evidence"]), a["rationale"], a["next_step"], a["product_id"], old["id"]))
                    c.execute("INSERT INTO history(action_id,ts,from_status,to_status,note) VALUES (?,?,?,?,?)", (old["id"], now(), "proposed", "proposed", "refreshed with the latest data"))
                continue
            c.execute("INSERT OR REPLACE INTO actions VALUES (?,?,?,?,?,?,?,?,?,?,?,?)", (a["id"], run_id, now(), a["title"], a["description"], a["priority"], a["score"],
                      json.dumps(a["evidence"]), a["rationale"], a["next_step"], a["status"], a["product_id"]))
            c.execute("INSERT INTO history(action_id,ts,from_status,to_status,note) VALUES (?,?,?,?,?)", (a["id"], now(), None, "proposed", "created by workflow"))

def list_actions(status=None, run_id=None):
    q, args = "SELECT * FROM actions WHERE 1=1", []
    if status: q += " AND status=?"; args.append(status)
    if run_id: q += " AND run_id=?"; args.append(run_id)
    with conn() as c: return [dict(r) for r in c.execute(q + " ORDER BY score DESC", args)]

def set_status(action_id, new, note=""):
    with conn() as c:
        row = c.execute("SELECT status FROM actions WHERE id=?", (action_id,)).fetchone()
        if row is None: raise KeyError(f"unknown action {action_id}")
        if new not in TRANSITIONS[row["status"]]: raise ValueError(f"cannot move {action_id} from '{row['status']}' to '{new}'")
        c.execute("UPDATE actions SET status=? WHERE id=?", (new, action_id))
        c.execute("INSERT INTO history(action_id,ts,from_status,to_status,note) VALUES (?,?,?,?,?)", (action_id, now(), row["status"], new, note or "changed by user"))

def retire_open_proposals(note="superseded: new data uploaded"):
    """When the business data is replaced, proposals made from the OLD data are no longer valid: mark them rejected (kept in the history). Approved/completed actions are left alone."""
    rows = list_actions(status="proposed")
    for r in rows: set_status(r["id"], "rejected", note)
    return len(rows)

def generate_proposals():
    """Build proposed actions straight from the data that is loaded now (no chat question needed, no AI wording -> fast).
    Runs the same specialists + coordinator + verifier as the 'most urgent actions' question. Returns how many proposals are open."""
    global USE_NARRATION, FEEDBACK_USE_LLM
    old = (USE_NARRATION, FEEDBACK_USE_LLM); USE_NARRATION = False; FEEDBACK_USE_LLM = False       # keep it instant
    try:
        sa, inv, fb = analyze_sales(), analyze_inventory(), analyze_feedback()
        co = coordinate(sa, inv, fb, search_docs("supplier lead time minimum order quantity expedited shipping reorder policy", 6))
        ver = verify(sa, inv, fb, None, co["actions"], co["summary"])
    finally:
        USE_NARRATION, FEEDBACK_USE_LLM = old
    if ver["passed"]: save_actions("RUN-" + uuid.uuid4().hex[:6], co["actions"])
    return len(list_actions(status="proposed"))

def edit_action(action_id, **fields):
    allowed = {"title", "description", "priority", "next_step"}                  # whitelist, so column names can never be injected
    if set(fields) - allowed: raise ValueError(f"cannot edit {set(fields) - allowed}")
    with conn() as c:
        row = c.execute("SELECT status FROM actions WHERE id=?", (action_id,)).fetchone()
        if row is None or row["status"] != "proposed": raise ValueError("only proposed actions can be edited")
        for k, v in fields.items(): c.execute(f"UPDATE actions SET {k}=? WHERE id=?", (v, action_id))

DRAFT_NOTE = "DRAFT - NOT SENT. Review and edit; this program never sends emails or places orders."

def make_drafts(inv, fb_rows):
    drafts = []
    by_supplier = {}
    for it in (inv or {}).get("items", []):
        if it["risk"] in ("critical", "high") and it["reorder_qty"]: by_supplier.setdefault(it["supplier"], []).append(it)
    for sup, items in by_supplier.items():
        lines = "\n".join(f"- {i['name']} ({i['product_id']}): stock {i['stock']:g}, draft quantity {i['reorder_qty']}" for i in items)
        drafts.append({"kind": "supplier", "recipient": sup, "subject": f"{BUSINESS}: please confirm lead time and availability",
                       "body": f"[{DRAFT_NOTE}]\n\nHello {sup} team,\nWe are reviewing replenishment and would like to confirm lead times, minimum order quantities and availability for:\n{lines}\nNo order has been placed.\n\n{BUSINESS} Operations"})
    for r in fb_rows[:3]:
        drafts.append({"kind": "customer", "recipient": f"customer of feedback {r.feedback_id}", "subject": f"About your feedback ({r.feedback_id})",
                       "body": f"[{DRAFT_NOTE}]\n\nThank you for your feedback about product {r.product_id}. We are sorry your experience fell short and a team member will contact you with next steps.\n\n{BUSINESS} Customer Care"})
    return drafts

def write_report(run_id, sales, inv, fb, co, warnings, ver):
    rid = "RPT-" + uuid.uuid4().hex[:6]; folder = f"{WORK}/reports"; os.makedirs(folder, exist_ok=True)
    L = [f"# {BUSINESS} Weekly Operations Report", "", "## Executive summary", co["summary"], ""]
    if sales: L += [f"## Sales - {sales['period']}", sales["narrative"], "", "| Top product | Revenue | Units |", "|---|---|---|"] + [f"| {t['name']} | {money(t['revenue'])} | {t['units']} |" for t in sales["top"]] + [""]
    if inv: L += ["## Inventory risk", inv["narrative"], "", "| Product | Risk | Stock | Days of stock | Lead time | Draft reorder |", "|---|---|---|---|---|---|"] + \
        [f"| {i['name']} | {i['risk']} | {i['stock']:g} | {i['coverage']} | {i['lead_time']:g} | {i['reorder_qty'] or '-'} |" for i in inv["items"] if i["risk"] != "low"] + ["", "**Assumptions**"] + [f"- {a}" for a in inv["assumptions"]] + [""]
    if fb: L += ["## Customer feedback", fb["narrative"], ""] + [f"- {h}" for h in fb["hypotheses"]] + [""]
    L += ["## Proposed actions (need approval)"] + [f"- **[{a['priority']}] {a['title']}** - {a['description']} Evidence: {', '.join(a['evidence'])}. Next: {a['next_step']}" for a in co["actions"]]
    L += ["", f"## Verification\nPassed: {ver['passed']} ({ver['checks']} checks). " + "; ".join(ver["errors"][:3])] + (["", "## Warnings"] + [f"- {w}" for w in warnings] if warnings else []) + ["", "---", DRAFT_NOTE]
    md = f"{folder}/{rid}.md"; open(md, "w").write("\n".join(L))
    c1, c2 = f"{folder}/{rid}_actions.csv", f"{folder}/{rid}_inventory_review.csv"
    pd.DataFrame([{**a, "evidence": " | ".join(a["evidence"])} for a in co["actions"]]).to_csv(c1, index=False)
    pd.DataFrame([i for i in (inv or {}).get("items", []) if i["risk"] != "low"]).to_csv(c2, index=False)
    low = DB["feedback"][DB["feedback"]["rating"] <= 2] if DB["feedback"] is not None else pd.DataFrame()
    drafts = make_drafts(inv, list(low.itertuples()))
    with conn() as c:
        c.execute("INSERT INTO reports VALUES (?,?,?,?,?,?)", (rid, now(), run_id, "Weekly Operations Report", md, f"{c1};{c2}"))
        for d in drafts: c.execute("INSERT INTO drafts VALUES (?,?,?,?,?,?,?)", ("DRF-" + uuid.uuid4().hex[:6], run_id, d["kind"], d["recipient"], d["subject"], d["body"], "draft"))
    return {"id": rid, "md": md, "csv": [c1, c2], "drafts": len(drafts)}

init_db(); print("Database ready:", DB_PATH)

## The Router and the LangGraph workflow
**What it does:** connects all eight agents into one flow-chart using **LangGraph**.

**Router** – reads your question and picks an *intent* using simple keywords, then decides which agents are needed. Only the needed agents run (a sales question does not wake the inventory agent):

| Intent (example question) | Agents that run |
|---|---|
| `sales` – "best-selling products last month?" | Sales |
| `inventory` – "which products risk running out?" | Inventory + Documents → Coordinator |
| `feedback` – "what are customers complaining about?" | Feedback |
| `documents` – "what do supplier terms say about lead times?" | Documents |
| `priorities` – "three most urgent actions?" | Sales + Inventory + Feedback + Documents (**in parallel**) → Coordinator → Reporter |
| `report` – "generate a weekly operations report" | same as priorities, and writes the report files |

A very short follow-up like *"why?"* re-uses the previous intent (the **conversation memory**).

**The graph**
* **State** = one shared dictionary that every step reads from and adds to.
* **Nodes** = the agents. Each is wrapped by `step(...)`, which times it and **catches errors**, so one broken agent never crashes the whole run.
* **Edges** = the arrows. *Conditional* edges choose the next step (e.g. "verification failed → go to `correct` once").
* The graph ends at `respond`, which writes the final answer and saves the run to SQLite. Actions stay `proposed` until a human decides.

**You should see:** `Workflow ready`.

In [ ]:
from langgraph.graph import StateGraph, START, END

INTENTS = [("report", r"(?:generate|create|write|make|produce|prepare|build)\W+(?:\w+\W+){0,3}report|weekly (?:operations )?report|operations report"),
           ("priorities", r"urgent|priorit|top (?:three|3) actions?|business actions?|what should i do"),
           ("documents", r"supplier terms|policy|policies|terms say|document|contract|payment terms|minimum order"),
           ("inventory", r"stock|inventory|reorder|run(?:ning)? out|lead time|at risk|\brisk\b|\bsafe\b|overstock\w*"),
           ("feedback", r"complain|customer|feedback|review|rating"),
           ("sales", r"sold|sell|revenue|sales|best[- ]selling|orders?|units?|underperform\w*|slow[- ]mov\w+|worst (?:products?|items?)|bottom \d* ?products?")]
ALL4 = ["sales", "inventory", "feedback", "docs"]
PLAN = {"sales": ["sales"], "inventory": ["inventory", "docs"], "feedback": ["feedback"], "documents": ["docs"],
        "priorities": ALL4, "report": ALL4, "general": []}
COORD_INTENTS = {"inventory", "priorities", "report"}      # only these need the Coordinator
MEMORY = []                                                 # conversation memory: [(intent, [product ids])]
OUT_OF_CONTEXT_MSG = ("⚠️ Sorry, the information you asked for is **out of context** for this model. "
                      "I only answer questions about Nova Retail's sales, inventory, customer feedback, supplier/policy documents, "
                      "priorities and reports. Try: *which products risk running out of stock?*")
GREETING = r"^\s*(hi|hello|hey|help|thanks|thank you)\b"
NEG_STOCK = r"(?:\bnot\b|n't\b|\bnever\b|\bno\b).{0,25}\b(?:run(?:ning)? out|at risk|low|short)\b|\b(?:safe|healthy|well[- ]stocked|plenty|overstock\w*)\b"   # the OPPOSITE of the stock-risk question
SELL_LOW = r"\b(?:least|fewest|lowest|worst|slowest|weakest|poorest|bottom|underperform\w*|slow[- ]mov\w+)\b"   # "least / worst selling" -> reverse the sales ranking
LEAST = r"\b(least|fewest|lowest|smallest|rarest)\b"           # "complaining about LEAST" -> reverse the ordering
FOLLOW_UP = r"^(why|how come|what about|and |tell me more|more\b|explain|which one|those|them|that\b|it\b)"

def route_question(text):
    t = text.lower()
    intent = next((name for name, rx in INTENTS if re.search(rx, t)), "general")
    ents = [e.upper() for e in re.findall(r"\bP\d{3}\b", text, flags=re.I)]
    if intent == "general" and ents: intent = "inventory"          # a product ID with no other keyword -> stock question
    if intent == "general" and MEMORY and len(t.split()) <= 5 and re.search(FOLLOW_UP, t.strip()):   # short follow-up like "why?"
        intent, ents = MEMORY[-1][0], ents or MEMORY[-1][1]
    return intent, PLAN[intent], ents

from typing import Literal, Optional

PLAN_KEYS = ["topic", "show", "metric", "order", "limit", "period", "products", "view"]
DEFAULT_PLAN = {"topic": "general", "show": "summary", "metric": "revenue", "order": "highest", "limit": 5, "period": None, "products": [], "view": "at_risk"}
LAST_PLAN = {}                                              # the previous question's plan, so follow-ups like "what about units" can reuse it
PLAN_CACHE = {}
def prev_plan(): return LAST_PLAN if MEMORY else {}         # "Clear conversation" empties MEMORY, which also forgets the previous plan
FOLLOW_UP_EXTRA = r"^(now|also|instead|only|just|by|sort|rank|show)\b"
def is_followup(text):
    """A SHORT question that builds on the previous one ('what about units', 'now by revenue'). Everything else is a fresh question."""
    t = text.lower().strip()
    return bool(prev_plan()) and len(t.split()) <= 7 and bool(re.search(FOLLOW_UP, t) or re.search(FOLLOW_UP_EXTRA, t))

class Plan(BaseModel):                                      # the exact shape we accept from the AI planner (anything else is rejected)
    topic: Literal["sales", "inventory", "feedback", "documents", "priorities", "report", "general"]
    show: Literal["summary", "ranking"] = "summary"
    metric: Literal["revenue", "units", "orders"] = "revenue"
    order: Literal["highest", "lowest"] = "highest"
    limit: int = Field(default=5, ge=1, le=8)
    period: Optional[str] = None
    products: list[str] = []
    view: Literal["at_risk", "safe"] = "at_risk"

PLAN_SYSTEM = """You turn a business question about the shop "Nova Retail" into ONE JSON query plan. Reply with JSON only.
Fields:
- topic: sales | inventory | feedback | documents | priorities | report | general
  (sales = revenue, units, orders, products sold; inventory = stock, running out, reorder; feedback = customer complaints and reviews;
   documents = supplier terms, policies, contracts; priorities = most urgent actions; report = ONLY when asked to generate or write a report;
   general = anything unrelated to the shop)
- show: "ranking" if the question wants products ordered by something (best, worst, top, least, most, which product...), else "summary"
- metric: revenue | units | orders (what products are ranked by; "units", "quantity", "how many sold" = units)
- order: "highest" or "lowest" (least, worst, fewest, slowest, lowest = lowest)
- limit: how many items, 1 to 8 (default 5)
- period: a phrase like "last month", "last 14 days", "last week", or null
- products: product IDs such as "P002" mentioned in the question, otherwise []
- view: inventory only: "safe" when asking which products are NOT at risk / safe / well stocked, otherwise "at_risk"
If the question is a short follow-up ("what about units?", "and the worst?"), copy the previous plan and change only what the follow-up changes.
Examples:
Q: what are my least selling products? -> {"topic":"sales","show":"ranking","metric":"revenue","order":"lowest","limit":5,"period":null,"products":[],"view":"at_risk"}
Q: which product sold the most units last month? -> {"topic":"sales","show":"ranking","metric":"units","order":"highest","limit":1,"period":"last month","products":[],"view":"at_risk"}
Q: how are sales doing? -> {"topic":"sales","show":"summary","metric":"revenue","order":"highest","limit":5,"period":null,"products":[],"view":"at_risk"}
Q: which products are not running out of stock? -> {"topic":"inventory","show":"summary","metric":"revenue","order":"highest","limit":5,"period":null,"products":[],"view":"safe"}
Q: why is P002 at risk? -> {"topic":"inventory","show":"summary","metric":"revenue","order":"highest","limit":5,"period":null,"products":["P002"],"view":"at_risk"}
Q: what are customers complaining about least? -> {"topic":"feedback","show":"summary","metric":"revenue","order":"lowest","limit":5,"period":null,"products":[],"view":"at_risk"}
Q: what is the capital of France? -> {"topic":"general","show":"summary","metric":"revenue","order":"highest","limit":5,"period":null,"products":[],"view":"at_risk"}
Follow-up: previous plan {"topic":"sales","show":"ranking","metric":"revenue","order":"lowest","limit":5,"period":null,"products":[],"view":"at_risk"}, question "what about units" -> {"topic":"sales","show":"ranking","metric":"units","order":"lowest","limit":5,"period":null,"products":[],"view":"at_risk"}"""

def rule_plan(text):
    """Keyword fallback: used when the AI is off or its answer is invalid. It also understands short follow-ups."""
    t = text.lower().strip()
    intent, _, ents = route_question(text)
    last = prev_plan()
    follow = is_followup(text)
    base = {**DEFAULT_PLAN, **(last if follow else {})}
    topic = intent if intent != "general" else (base["topic"] if follow else "general")
    metric = ("units" if re.search(r"\bunits?\b|quantit\w*|volume|pieces", t) else
              "orders" if re.search(r"\borders?\b", t) else
              "revenue" if re.search(r"revenue|income|money|\bsales\b", t) else base["metric"])
    low = re.search(SELL_LOW if topic == "sales" else LEAST, t); high = re.search(r"\b(best|top|most|highest|biggest|largest)\b", t)
    order = "lowest" if low else "highest" if high else base["order"]
    ranking = bool(re.search(r"\b(best|top|worst|least|most|lowest|highest|fewest|rank\w*|which products?|selling|sellers?|slow\w*|underperform\w*|bottom)\b", t)) or (follow and base["show"] == "ranking")
    m = re.search(r"\b(?:top|bottom|best|worst|least|most|lowest|highest)\s+(\d+|one|two|three|four|five)\b", t)
    limit = ({"one": 1, "two": 2, "three": 3, "four": 4, "five": 5}.get(m.group(1)) or int(m.group(1))) if m else base["limit"]
    pm = re.search(r"(?:last|past)\s+\d{1,3}\s+days?|last month|this month|this week|last week|weekly|monthly|\bweek\b", t)
    return {"topic": topic, "show": "ranking" if ranking else "summary", "metric": metric, "order": order, "limit": max(1, min(8, limit)),
            "period": pm.group(0) if pm else base["period"], "products": ents or base["products"],
            "view": "safe" if re.search(NEG_STOCK, t) else (base["view"] if topic == "inventory" else "at_risk"), "source": "keyword rules"}

def llm_plan(text):
    """Ask the AI for a query plan. Returns a validated plan, or None (no AI / invalid answer) so the keyword rules take over."""
    last = prev_plan(); prev = json.dumps({k: last[k] for k in PLAN_KEYS}) if (last and is_followup(text)) else "none"   # only a real follow-up gets to see the previous plan
    key = (text.strip().lower(), prev)
    if key in PLAN_CACHE: return PLAN_CACHE[key]
    raw = llm_chat([{"role": "system", "content": PLAN_SYSTEM}, {"role": "user", "content": f"Previous plan: {prev}\nQ: {text} ->"}], 150)
    try:
        d = extract_json(raw)
        if not isinstance(d, dict): return None
        if str(d.get("period")).lower() in ("null", "none", ""): d["period"] = None
        d["limit"] = max(1, min(8, int(d.get("limit") or 5)))
        d["products"] = [str(p).upper() for p in (d.get("products") or []) if re.fullmatch(r"P\d{3}", str(p), flags=re.I)]
        plan = {**Plan(**d).model_dump(), "source": "AI planner"}
    except Exception:
        return None
    PLAN_CACHE[key] = plan; return plan

def make_plan(text):
    """The AI understands the question; the keyword rules are the safety net."""
    rp, lp = rule_plan(text), llm_plan(text)
    ids = [e.upper() for e in re.findall(r"\bP\d{3}\b", text, flags=re.I)]
    if lp is None or (lp["topic"] == "general" and rp["topic"] != "general") or (lp["topic"] == "report" and rp["topic"] != "report"):
        return rp                                           # AI off / invalid / missed an obvious topic / tried to write files without being asked
    period = lp["period"] if lp["period"] and lp["period"].lower() in text.lower() else rp["period"]   # a period must be IN the question (or inherited by a real follow-up)
    return {**lp, "period": period, "products": rp["products"]}                                         # product IDs: only those typed now (or inherited by a follow-up)

class State(TypedDict, total=False):
    run_id: str; request: str; intent: str; agents: list; entities: list; period: tuple; started: float; retried: bool
    sales: dict; inventory: dict; feedback: dict; rag: dict; doc_hits: list; coord: dict; verification: dict; report: dict
    steps: Annotated[list, operator.add]        # the "+" means parallel agents can all append to the log safely
    warnings: Annotated[list, operator.add]
    answer: str; status: str; direction: str; plan: dict

def step(name):
    """Wrap an agent function: time it, record a log line, and turn crashes into warnings instead of stopping the run."""
    def deco(fn):
        def run(state):
            t0 = time.time()
            try:
                update, summary = fn(state)
                return {**update, "steps": [{"agent": name, "status": "ok", "seconds": round(time.time() - t0, 2), "summary": summary}]}
            except Exception as e:
                msg = f"{type(e).__name__}: {str(e)[:150]}"
                return {"steps": [{"agent": name, "status": "error", "seconds": round(time.time() - t0, 2), "summary": "failed", "error": msg}],
                        "warnings": [f"{name} failed ({msg}); continuing without it."]}
        return run
    return deco

@step("router")
def n_route(s):
    plan = make_plan(s["request"]); intent = plan["topic"]; agents, ents = PLAN[intent], plan["products"]
    period = parse_period(s["request"], DB["as_of"]) if DB["as_of"] is not None else None
    if period and not period[3] and plan["period"]: period = parse_period(plan["period"], DB["as_of"])   # a period carried over from the previous question
    if intent == "report" and period and not period[3]:            # a weekly report looks at the last 7 days
        end = DB["as_of"]; period = (end - pd.Timedelta(days=6), end, "last 7 days", True)
    return ({"intent": intent, "agents": agents, "entities": ents, "period": period, "plan": plan, "direction": "least" if plan["order"] == "lowest" else "most"},
            f"intent={intent} (read by {plan['source']}); agents={agents or 'none'}")

@step("data_collector")
def n_collect(s):
    return {"warnings": list(DB["warnings"])}, " | ".join(f"{r['kind']}: {r['clean']}/{r['rows']} valid" for r in DB["reports"].values()) or "no data loaded"

@step("sales_analyst")
def n_sales(s):
    p = s.get("period"); f = analyze_sales(p[0], p[1], p[2]) if p else analyze_sales()
    pl = s.get("plan") or {}
    if f.get("start") and pl.get("show") == "ranking":
        f["rank"] = rank_products(pl["metric"], pl["order"], pl["limit"], pd.Timestamp(f["start"]), pd.Timestamp(f["end"]), pl.get("products") or None)
        f["rank_meta"] = {"metric": pl["metric"], "order": pl["order"]}
    return {"sales": f}, f"{f['period']}: revenue {money(f['revenue'])}, {f['orders']} orders"

@step("inventory_analyst")
def n_inventory(s):
    f = analyze_inventory(); c = f["counts"]
    return {"inventory": f}, f"{c.get('critical', 0)} critical, {c.get('high', 0)} high, {c.get('medium', 0)} medium"

@step("feedback_analyst")
def n_feedback(s):
    p = s.get("period"); use = p and p[3] and s["intent"] != "report"
    f = analyze_feedback(p[0], p[1]) if use else analyze_feedback()
    return {"feedback": f}, f"{f['total']} comments, method={f['method']}"

@step("document_researcher")
def n_docs(s):
    if s["intent"] == "documents":
        r = ask_docs(s["request"]); return {"rag": r, "doc_hits": r["hits"]}, "answered with citations" if r["sufficient"] else "insufficient evidence"
    hits = search_docs("supplier lead time minimum order quantity expedited shipping reorder policy", 6)
    return {"doc_hits": hits}, f"{len(hits)} supplier/policy passage(s) found"

@step("operations_coordinator")
def n_coordinate(s):
    c = coordinate(s.get("sales"), s.get("inventory"), s.get("feedback"), s.get("doc_hits")); return {"coord": c}, f"{len(c['actions'])} proposed action(s)"

@step("verifier")
def n_verify(s):
    co = s.get("coord") or {}
    v = verify(s.get("sales"), s.get("inventory"), s.get("feedback"), s.get("rag"), co.get("actions"), co.get("summary"))
    return {"verification": v}, f"{'PASSED' if v['passed'] else 'FAILED'}: {v['checks']} checks, {len(v['errors'])} problem(s)"

@step("correction")
def n_correct(s):
    """Safe repair: replace AI wording with the verified template, drop evidence that cannot be found."""
    upd = {"retried": True}
    for key in ("sales", "inventory", "feedback"):
        if s.get(key): upd[key] = {**s[key], "narrative": s[key]["fallback"], "narrative_source": "template"}
    if s.get("rag") and s["rag"]["hits"] and s["rag"]["sufficient"]: upd["rag"] = quote_answer(s["rag"]["hits"])
    if s.get("coord"):
        co = dict(s["coord"]); co["summary"] = co["summary_fallback"]
        co["actions"] = [{**a, "evidence": [e for e in a["evidence"] if evidence_ok(e)]} for a in co["actions"]]
        upd["coord"] = co
    return upd, "replaced AI text with verified templates and removed unverifiable evidence"

@step("reporter")
def n_report(s):
    co = s["coord"]; run_id = s["run_id"]
    if s["intent"] == "report":
        rep = write_report(run_id, s.get("sales"), s.get("inventory"), s.get("feedback"), co, s.get("warnings", []), s["verification"])
        save_actions(run_id, co["actions"]); return {"report": rep}, f"report {rep['id']} saved; {rep['drafts']} draft(s); {len(co['actions'])} action(s) await approval"
    save_actions(run_id, co["actions"]); return {"report": {}}, f"{len(co['actions'])} action(s) saved as 'proposed'"

def compose(s):
    """Build the final answer text from the (verified) findings."""
    I, ent = s["intent"], set(s.get("entities", [])); out = []
    sa, inv, fb, rag, co = s.get("sales"), s.get("inventory"), s.get("feedback"), s.get("rag"), s.get("coord")
    plan = {**DEFAULT_PLAN, **(s.get("plan") or {})}
    if I == "sales" and sa:
        if sa.get("rank"):                                                 # the question asked for a ranking: lead sentence + table, any metric / direction
            m = plan["metric"]; fmt = money if m == "revenue" else (lambda v: f"{v:,}")
            what = {"revenue": "revenue", "units": "units sold", "orders": "number of orders"}[m]
            lead = f"{sa['period']}, ranked by {what} ({plan['order']} first): " + "; ".join(f"{t['name']} ({fmt(t[m])})" for t in sa["rank"][:3]) + "."
            out += [lead, "| Product | Revenue | Units | Orders |\n|---|---|---|---|\n" + "\n".join(f"| {t['name']} ({t['product_id']}) | {money(t['revenue'])} | {t['units']:,} | {t['orders']:,} |" for t in sa["rank"])]
        else:
            out += [sa["narrative"], "| Top product | Revenue | Units |\n|---|---|---|\n" + "\n".join(f"| {t['name']} ({t['product_id']}) | {money(t['revenue'])} | {t['units']} |" for t in sa["top"])]
        out += [f"_Note: {n}_" for n in sa["notes"]]
    elif I == "inventory" and inv:
        known = {i["product_id"] for i in inv["items"]}
        if ent and not (ent & known):
            return f"I couldn't find {', '.join(sorted(ent))} in the inventory data."
        safe_view = bool(not ent and plan["view"] == "safe")   # "which products are NOT running out?" -> the opposite list
        if safe_view:
            rows = sorted([i for i in inv["items"] if i["risk"] in ("low", "medium", "unknown")], key=lambda i: -(i["coverage"] if i["coverage"] is not None else -1))
            names = ", ".join(f"{i['name']} ({i['coverage']} days of stock)" if i["coverage"] is not None else i["name"] for i in rows[:5])
            lead = (f"{len(rows)} of {len(inv['items'])} products are not at risk of running out before a restock could arrive. Safest: {names}. "
                    "A 'medium' product is below its reorder level but still has enough cover.") if rows else "Every product is at risk of running out - none look safe right now."
        else:
            rows = [i for i in inv["items"] if (i["product_id"] in ent if ent else i["risk"] != "low")]
            if ent:                                                            # product-specific: sentence built from the verified numbers
                lead = " ".join(f"{i['name']} ({i['product_id']}) is **{i['risk']}** risk: {i['stock']:g} in stock, "
                                + (f"{i['coverage']} days of stock vs a {i['lead_time']:g}-day lead time." if i["coverage"] is not None else "no recent sales to estimate coverage.")
                                + (f" Demand is {i['growth_pct']:+.0f}% over the last 14 days." if i["growth_pct"] is not None else "")
                                for i in rows)
            else:
                lead = inv["narrative"]
        out += [lead, "| Product | Risk | Stock | Days of stock | Lead time | Draft reorder |\n|---|---|---|---|---|---|\n" + "\n".join(f"| {i['name']} ({i['product_id']}) | {i['risk']} | {i['stock']:g} | {i['coverage']} | {i['lead_time']:g} | {i['reorder_qty'] or '-'} |" for i in rows),
                "**Assumptions:** " + " ".join(inv["assumptions"])]
    elif I == "feedback" and fb:
        least = plan["order"] == "lowest"
        themes = [t for t in fb["themes"] if t["theme"] != "other"]       # "other" is mostly praise, not a complaint
        if least and themes:
            themes = sorted(themes, key=lambda t: (t["negative"], t["count"]))   # fewest negative comments first
            lead = f"Least complained about: **{themes[0]['theme']}** ({themes[0]['negative']} negative of {themes[0]['count']} mentions)."
        else:
            lead = fb["narrative"]; themes = fb["themes"]
        out += [lead, "| Theme | Mentions | Negative | Feedback IDs |\n|---|---|---|---|\n" + "\n".join(f"| {t['theme']} | {t['count']} | {t['negative']} | {', '.join(t['ids'][:4])}{'...' if len(t['ids']) > 4 else ''} |" for t in themes)]
        if fb["hypotheses"] and not least: out.append("**Hypotheses (unverified):** " + " ".join(fb["hypotheses"]))
    elif I == "documents" and rag and not rag["sufficient"]:
        return OUT_OF_CONTEXT_MSG + "\n\n_No matching passage was found in the documents._"
    elif I == "documents" and rag:
        out += [rag["answer"]] + (["Sources: " + ", ".join(f"[{a} p.{b}]" for a, b in rag["citations"])] if rag["citations"] else [])
    elif I in ("priorities", "report") and co:
        if I == "report" and s.get("report"):
            r = s["report"]; out.append(f"Weekly report **{r['id']}** created: `{r['md']}` (+ 2 CSV files). {r['drafts']} follow-up draft(s) were written - **nothing was sent or ordered.**")
        out.append("**Most urgent actions** (proposals; approve or reject them in the Actions tab):")
        out += [f"{n}. **{a['title']}** [{a['priority']}]\n   - {a['description']}\n   - Why: {a['rationale']}\n   - Evidence: {', '.join(a['evidence'])}\n   - Next step: {a['next_step']}" for n, a in enumerate(co["actions"][:3], 1)] or ["No urgent risks found."]
    elif I == "general" and not re.search(GREETING, s["request"].lower()):
        return OUT_OF_CONTEXT_MSG                                          # anything outside the business topics
    else:
        out.append("I can analyse sales, inventory risk, customer feedback and your documents. Try: *best-selling products last month*, *which products risk running out of stock?*, *what are customers complaining about most?*, *what are my three most urgent business actions?*, *what do supplier terms say about reorder lead times?* or *generate a weekly operations report*.")
    src = plan.get("source", "keyword rules")                                  # tell the user how the question was understood (makes mistakes visible)
    if I == "sales" and sa and sa.get("rank"): out.append(f"_Understood as: rank products by {plan['metric']}, {plan['order']} first, {sa['period']} (read by {src})._")
    elif I == "inventory" and inv and not ent: out.append(f"_Understood as: {'products NOT at risk' if plan['view'] == 'safe' else 'products at risk of running out'} (read by {src})._")
    elif I == "feedback" and fb: out.append(f"_Understood as: complaint themes, {plan['order']} first (read by {src})._")
    v = s.get("verification")
    if v: out.append(f"_Verification: {'passed' if v['passed'] else 'FAILED - please review'} ({v['checks']} checks)._" + "".join(f"\n- Unresolved: {e}" for e in v["errors"][:3]))
    warns = [w for w in dict.fromkeys(s.get("warnings", [])) if "rejected" not in w or I == "report"]
    if warns: out.append("**Warnings:** " + " ".join(f"({i+1}) {w}" for i, w in enumerate(warns[:3])))
    return "\n\n".join(x for x in out if x)

@step("respond")
def n_respond(s):
    answer = compose(s); v = s.get("verification") or {"passed": True}
    status = "needs_human_review" if not v["passed"] else "awaiting_approval" if s["intent"] in ("priorities", "report") and (s.get("coord") or {}).get("actions") else "completed"
    with conn() as c:
        c.execute("INSERT OR REPLACE INTO runs VALUES (?,?,?,?,?,?,?,?)", (s["run_id"], now(), s["request"], s["intent"], status, round(time.time() - s["started"], 2), json.dumps(s["steps"]), json.dumps(v)))
    if s["intent"] != "general":
        MEMORY.append((s["intent"], s.get("entities", []))); LAST_PLAN.clear(); LAST_PLAN.update(s.get("plan") or {})
    return {"answer": answer, "status": status}, f"run saved ({status})"

def after_verify(s):
    if not s["verification"]["passed"] and not s.get("retried"): return "correct"             # one repair attempt only (bounded loop)
    return "reporter" if s["intent"] in ("report", "priorities") and s.get("coord") else "respond"

g = StateGraph(State)
for name, fn in [("route", n_route), ("collect", n_collect), ("sales_agent", n_sales), ("inventory_agent", n_inventory), ("feedback_agent", n_feedback),
                 ("docs_agent", n_docs), ("coordinate", n_coordinate), ("verify", n_verify), ("correct", n_correct), ("reporter", n_report), ("respond", n_respond)]:
    g.add_node(name, fn)
AGENT_NODE = {"sales": "sales_agent", "inventory": "inventory_agent", "feedback": "feedback_agent", "docs": "docs_agent"}
g.add_edge(START, "route"); g.add_edge("route", "collect")
g.add_conditional_edges("collect", lambda s: [AGENT_NODE[a] for a in s["agents"]] or ["verify"], list(AGENT_NODE.values()) + ["verify"])   # fan-out (parallel)
for node in AGENT_NODE.values():
    g.add_conditional_edges(node, lambda s: "coordinate" if s["intent"] in COORD_INTENTS else "verify", ["coordinate", "verify"])
g.add_edge("coordinate", "verify")
g.add_conditional_edges("verify", after_verify, ["correct", "reporter", "respond"])
g.add_edge("correct", "verify"); g.add_edge("reporter", "respond"); g.add_edge("respond", END)
GRAPH = g.compile()

def new_state(question): return {"run_id": "RUN-" + uuid.uuid4().hex[:6], "request": question, "steps": [], "warnings": [], "retried": False, "started": time.time()}
def ask(question):
    """Run the whole multi-agent workflow for one question and return the final state."""
    return GRAPH.invoke(new_state(question))
print("Workflow ready. Nodes:", [n for n in GRAPH.get_graph().nodes if n not in ("__start__", "__end__")])

## Try it!
**What it does:** asks the six example questions. For the first one it also prints the **agent log** (which agent ran, how long it took, what it found) so you can see the flow.

**You should see:** for each question, an answer with numbers/tables, a `Verification: passed` line and, for actions, evidence references such as `inventory:P002` or `doc:supplier_terms.pdf p.1`.

💡 Try your own questions by editing the list. Product IDs (e.g. "why is P002 at risk?") work as follow-ups too.

In [ ]:
QUESTIONS = ["What were my best-selling products last month?", "Which products risk running out of stock?",
             "What are customers complaining about most?", "What are my three most urgent business actions?",
             "What do supplier terms say about reorder lead times?", "Generate a weekly operations report"]
for i, q in enumerate(QUESTIONS):
    state = ask(q)
    print("=" * 90, f"\nQ: {q}   [{state['status']}]")
    if i == 0:
        print(pd.DataFrame(state["steps"])[["agent", "status", "seconds", "summary"]].to_string(index=False), "\n")
    print(state["answer"][:900])

## Self-checks (the tests)
**What it does:** runs ~20 small automatic checks so you can trust the system. A check is just *"I expect this result; does the code agree?"*

They cover: data validation, the revenue/average-order maths (against hand-calculated numbers), the stock-risk rules, the feedback classifier, the number-checking logic, the safe status changes in the database, document search + "I don't know" behaviour, routing, parallel agents, **an AI that lies** (the Verifier must catch the invented number), and **an agent that crashes** (the run must continue).

The checks switch the AI **off** while they run (and switch it back on at the end), so they finish in seconds and need no GPU. ✅ = passed, ❌ = failed. Only results that actually happened are shown.

**You should see:** a list of ✅ lines and a final count.

In [ ]:
results = []
_true_llm = llm_chat                                     # keep the real AI aside
llm_chat = lambda messages, max_new_tokens=300: None     # "no AI" mode: checks use the verified templates, so they run in seconds
def clear_caches(): NARRATE_CACHE.clear()
clear_caches()
def check(name, condition):
    results.append(bool(condition)); print("✅" if condition else "❌", name)

# --- data validation (Agent 1)
bad = pd.DataFrame({"order_id": ["1", "2", "3", "4", "5", "5"], "date": ["2026-01-01", "2026-13-45", "2026-01-02", "2026-01-02", "2026-01-03", "2026-01-03"],
                    "product_id": ["A", "A", "A", "", "B", "B"], "product_name": ["a"] * 6, "quantity": ["2", "1", "-3", "1", "1", "1"], "unit_price": ["10", "10", "10", "10", "5", "5"]})
c_, r_, rep = validate(bad, "sales")
check("validation keeps only the 2 good rows (bad date, negative qty, missing id, duplicate are rejected)", len(c_) == 2 and len(r_) == 4)
check("validation explains why rows were rejected", {"bad_date", "bad_quantity", "missing_value", "duplicate"} <= set(rep["issues"]))
check("missing columns are reported, not crashed on", validate(pd.DataFrame({"x": [1]}), "sales")[2]["usable"] is False)
# --- sales maths (Agent 2) - hand calculated: 2x10 + 1x5.5 + 3x10 + 4x2.25 + 2x5.5 = 75.5 over 3 orders
tiny = pd.DataFrame({"order_id": ["O1", "O1", "O2", "O3", "O3"], "date": pd.to_datetime(["2026-01-01"] * 2 + ["2026-01-02", "2026-01-03", "2026-01-03"]),
                     "quantity": [2, 1, 3, 4, 2], "unit_price": [10, 5.5, 10, 2.25, 5.5]}); tiny["line_revenue"] = tiny.quantity * tiny.unit_price
m = sales_metrics(tiny, pd.Timestamp("2026-01-01"), pd.Timestamp("2026-01-03"))
check("revenue = 75.5, units = 12, orders = 3", m["revenue"] == 75.5 and m["units"] == 12 and m["orders"] == 3)
check("average order value = revenue / orders", abs(m["aov"] - 75.5 / 3) < 0.01)
check("no orders -> no average (no divide-by-zero)", sales_metrics(tiny, pd.Timestamp("2030-01-01"), pd.Timestamp("2030-01-02"))["aov"] is None)
check("'last month' = most recent complete month", parse_period("last month", pd.Timestamp("2026-09-30"))[2] == "September 2026")
# --- inventory rules (Agent 3)
check("coverage shorter than lead time = critical", classify_risk(10, 5, 14, 1.0)[0] == "critical")
check("coverage inside the safety buffer = high", classify_risk(15, 5, 14, 1.0)[0] == "high")
check("below reorder level but long coverage = medium", classify_risk(4, 5, 7, 0.1)[0] == "medium")
check("healthy stock = low; no history = unknown", classify_risk(40, 5, 14, 1.0)[0] == "low" and classify_risk(40, 5, 7, None)[0] == "unknown")
# --- feedback + number checker
check("keyword classifier finds delivery vs praise", keyword_classify("Parcel arrived late", 1)["themes"] == ["delivery"] and keyword_classify("Great product, fast shipping", 5)["themes"] == ["other"])
check("number checker accepts real numbers and flags invented ones", bad_numbers("Revenue was $35,974", {"revenue": 35974.46}) == [] and bad_numbers("Revenue was $999,999", {"revenue": 35974.46}) == [999999.0])
# --- database (Agent 8), using a temporary file so your real data is untouched
_real_db = DB_PATH; DB_PATH = tempfile.mktemp(suffix=".sqlite"); init_db()
act = {"id": "A1", "title": "t", "description": "d", "priority": "high", "score": 9, "evidence": ["inventory:P001"], "rationale": "r", "next_step": "n", "status": "proposed", "product_id": "P001"}
save_actions("RUN-T", [act]); set_status("A1", "approved")
try: set_status("A1", "proposed"); moved_back = True
except ValueError: moved_back = False
try: edit_action("A1", status="completed"); edit_ok = True
except ValueError: edit_ok = False
check("actions follow proposed -> approved -> completed; illegal moves are blocked", list_actions(status="approved") and not moved_back)
check("edits only allowed on proposed actions and whitelisted fields", not edit_ok)
save_actions("RUN-T2", [{**act, "id": "A2"}])
check("asking again does not create a duplicate open proposal", len(list_actions()) == 1)
save_actions("RUN-T3", [{**act, "id": "A3", "title": "t2", "description": "old numbers", "score": 5}])
save_actions("RUN-T4", [{**act, "id": "A4", "title": "t2", "description": "NEW numbers", "score": 7}])
_t2 = [r for r in list_actions() if r["title"] == "t2"]
check("re-proposing an open action refreshes its numbers instead of keeping stale ones", len(_t2) == 1 and _t2[0]["description"] == "NEW numbers" and _t2[0]["score"] == 7)
check("new data retires old proposals (approved ones are kept)", retire_open_proposals() == 1 and len(list_actions(status="approved")) == 1)
check("fresh proposals are generated from the loaded data (open, ready to approve)", generate_proposals() > 0 and all(a["status"] == "proposed" for a in list_actions(status="proposed")))
DB_PATH = _real_db; init_db()
# --- uploads: headers, reasons (Agent 1)
_fb = pd.DataFrame({"Review ID": ["F1", "F2", "F3"], "Review Date": ["2026-01-01"] * 3, "Stars": ["9", "", "4"], "Comment": ["Excellent value", "No rating was given", "ok"], "SKU": ["A", "A", "A"]})
c_, r_, rep_ = validate(_fb, "feedback")
check("other column names are understood (Stars->rating, Comment->feedback_text, SKU->product_id...)", rep_["usable"] and len(c_) == 1 and detect_kind(_fb)[0] == "feedback")
check("rating 9 is rejected as bad_rating and the explanation shows the value", "bad_rating" in r_.loc[r_.feedback_id == "F1", "reject_reason"].iloc[0] and "9" in r_.loc[r_.feedback_id == "F1", "reject_detail"].iloc[0])
check("a MISSING rating is reported once, as missing_value (not bad_rating twice)", r_.loc[r_.feedback_id == "F2", "reject_reason"].iloc[0] == "missing_value;")
check("a file with the wrong columns is reported, not guessed", validate(pd.DataFrame({"foo": [1]}), "sales")[2]["usable"] is False and detect_kind(pd.DataFrame({"foo": [1]}))[1] < 0.5)
check("dd/mm/yyyy dates are read day-first when the file proves it", list(parse_dates(pd.Series(["25/12/2026", "05/01/2026"])).dt.month) == [12, 1])
# --- documents (Agent 5)
check("document search returns the right page with a citation", ask_docs("standard lead time for Nile Tech Distribution")["citations"][0] == ("supplier_terms.pdf", 1))
check("unanswerable question -> INSUFFICIENT EVIDENCE (no invention)", "INSUFFICIENT EVIDENCE" in ask_docs("Who won the football world cup?")["answer"])
# --- workflow
names = lambda st: [x["agent"] for x in st["steps"]]
st = ask("What were my best-selling products last month?")
check("a sales question runs only the sales agent (no coordinator)", "sales_analyst" in names(st) and "operations_coordinator" not in names(st) and "inventory_analyst" not in names(st))
st = ask("What are my three most urgent business actions?")
check("priorities run all 4 specialists, then coordinator, then verifier", {"sales_analyst", "inventory_analyst", "feedback_analyst", "document_researcher"} <= set(names(st)) and names(st).index("operations_coordinator") < names(st).index("verifier"))
check("verification passes and new actions start as 'proposed'", st["verification"]["passed"] and all(a["status"] == "proposed" for a in st["coord"]["actions"]))
for q in ["Who won the football world cup in 2022?", "What is the capital of Australia?", "Write me a poem about cats"]:
    check(f"out-of-context question gets the saved message: {q!r}", "out of context" in ask(q)["answer"])
check("a greeting still gets the help text", "I can analyse" in ask("hello")["answer"])
ask("Which products risk running out of stock?")
check("a short unrelated question after a real one is NOT treated as a follow-up", "out of context" in ask("capital of France?")["answer"])
# a pretend AI that invents numbers
_real_llm = llm_chat; clear_caches()                     # clear first, otherwise a cached answer would hide the test
llm_chat = lambda messages, max_new_tokens=300: "Revenue reached $999,999.99 and 4,242 units shipped, up 77.7%."
st = ask("What were my best-selling products last month?")
llm_chat = _real_llm; clear_caches()
check("an AI that invents numbers is caught and replaced by the verified text", "correction" in names(st) and "999,999" not in st["answer"] and st["verification"]["passed"])
# an agent that crashes
_real_fb = analyze_feedback
def _boom(*a, **k): raise RuntimeError("feedback exploded")
analyze_feedback = _boom
st = ask("What are my three most urgent business actions?")
analyze_feedback = _real_fb
check("a crashing agent does not stop the run", any(x["status"] == "error" for x in st["steps"]) and bool(st["answer"]) and bool(st["coord"]["actions"]))
# --- two regressions found in a real run: (1) the AI planner copied an old period into a new question, (2) the AI stock summary contradicted the table
_keep = llm_chat; PLAN_CACHE.clear(); MEMORY.clear(); LAST_PLAN.clear(); _seen = []
def _stale_ai(messages, max_new_tokens=300):                  # a pretend planner that ALWAYS answers with an old period and an old product
    _seen.append(messages[-1]["content"])
    return '{"topic":"feedback","show":"summary","metric":"revenue","order":"highest","limit":5,"period":"last month","products":["P002"],"view":"at_risk"}'
llm_chat = _stale_ai
ask("What were my best-selling products last month?")         # gives the planner something to 'remember'
st = ask("What are customers complaining about most?")        # a new, unrelated question with no period and no product
llm_chat = _keep; PLAN_CACHE.clear(); MEMORY.clear(); LAST_PLAN.clear()
check("the planner cannot carry an old period or product into an unrelated question", st["plan"]["period"] is None and st["plan"]["products"] == [] and "Previous plan: none" in _seen[-1])
inv_now = analyze_inventory()
check("the stock summary lists every product under its true risk level (written from the data, not by the AI)",
      inv_now["narrative_source"] == "template" and all(f"{len(n)} {lvl} ({', '.join(n)})" in inv_now["narrative"]
                                                        for lvl in ("critical", "high", "medium") for n in [[i["name"] for i in inv_now["items"] if i["risk"] == lvl]] if n))
print(f"\n{sum(results)}/{len(results)} checks passed")
llm_chat = _true_llm; clear_caches()                     # put the real AI back for the rest of the notebook

## Evaluation (how good is it, measured)
**What it does:** measures quality against answers we know in advance and prints **only what it actually measured** – nothing is pre-filled.

| Metric | Compared against |
|---|---|
| Numeric accuracy | a tiny dataset with hand-calculated results |
| Stock-risk alerts: **precision / recall** | the risk we *designed* into the sample data (`stock ÷ true demand < lead time + 3`) |
| Feedback classification | 16 hand-labelled comments |
| Document search: **hit@3** and refusals | 8 questions with known answer pages + 2 unanswerable ones |
| Workflow success rate and time | the 6 example questions |

*Precision* = of the alerts raised, how many were right. *Recall* = of the real risks, how many were found.

⚠️ The sample data is synthetic, and the stock labels come from how it was built, so high scores show the rules match the design – not real-world accuracy. To see how much the AI helps, run this cell with and without the model (`USE_LLM`).

**You should see:** a short list of measured values.

In [ ]:
# 1) numeric accuracy
numeric_ok = (m["revenue"] == 75.5) + (m["units"] == 12) + (m["orders"] == 3) + (abs(m["aov"] - 75.5 / 3) < 0.01)
print(f"numeric accuracy: {numeric_ok}/4")
# 2) stock-risk alerts versus the designed truth
truth = {p[0]: p[8] / (p[6] * p[7]) < p[5] + SAFETY_DAYS for p in PRODUCTS}
pred = {i["product_id"]: i["risk"] in ("critical", "high") for i in analyze_inventory()["items"]}
tp = sum(truth[k] and pred[k] for k in truth); fp = sum(not truth[k] and pred[k] for k in truth); fn = sum(truth[k] and not pred[k] for k in truth)
print(f"stock-risk alerts: precision={tp/max(tp+fp,1):.2f} recall={tp/max(tp+fn,1):.2f}  (tp={tp}, fp={fp}, fn={fn})")
# 3) feedback classification (uses the AI if loaded, else keywords)
LABELLED = [("The courier left my parcel in the rain and it came a week late", "delivery"), ("Shipping took forever", "delivery"), ("Stopped working after two weeks, clearly defective", "quality"),
            ("It feels flimsy and the hinge broke", "quality"), ("Way too expensive for what it is", "price"), ("Overpriced compared with the shop next door", "price"),
            ("Always sold out when I try to order", "availability"), ("Out of stock again, any restock date?", "availability"), ("Support ignored my refund request", "customer_service"),
            ("The staff were rude when I asked about a return", "customer_service"), ("Love it, works exactly as described", "other"), ("Excellent product, will recommend", "other"),
            ("Really happy with this purchase", "other"), ("Perfect size and colour", "other"), ("Delivery was delayed by a week", "delivery"), ("Cheap plastic, scratched immediately", "quality")]
rows = [(f"E{i}", t) for i, (t, _) in enumerate(LABELLED)]
got = {}
if llm_model is not None and FEEDBACK_USE_LLM:
    for i in range(0, len(rows), 8): got.update(classify_with_llm(rows[i:i + 8]))
hits_ = sum(((got.get(f"E{i}") or {}).get("themes") or keyword_classify(t, None)["themes"])[0] == lab for i, (t, lab) in enumerate(LABELLED))
print(f"feedback classification (first-theme accuracy, {'AI + keyword fallback' if got else 'keywords only'}): {hits_}/{len(LABELLED)}")
# 4) document search
QA = [("standard lead time for Nile Tech Distribution", ("supplier_terms.pdf", 1)), ("minimum order quantity at Delta Home Supplies", ("supplier_terms.pdf", 1)), ("payment terms of BrightSource Wholesale", ("supplier_terms.pdf", 2)),
      ("lead time at GreenLeaf Goods", ("supplier_terms.pdf", 2)), ("who approves purchase orders above 2,500", ("business_policy.pdf", 1)), ("safety buffer added to lead times", ("business_policy.pdf", 1)),
      ("return window for customers", ("business_policy.pdf", 2)), ("first reply time for customer messages", ("business_policy.pdf", 2))]
hit3 = sum(any((h["source"], h["page"]) == gold for h in search_docs(q, 3)) for q, gold in QA)
refused = sum(not ask_docs(q)["sufficient"] for q in ["Who won the football world cup in 2022?", "What is the capital of Australia?"])
print(f"document search: hit@3={hit3}/{len(QA)}, correct refusals={refused}/2  (embedder: {EMBEDDER})")
# 5) workflow
t0 = time.time(); runs = [ask(q) for q in QUESTIONS]
print(f"workflow: {sum(r['status'] != 'needs_human_review' and not any(x['status']=='error' for x in r['steps']) for r in runs)}/{len(runs)} runs succeeded, "
      f"{sum(r['verification']['passed'] for r in runs)}/{len(runs)} verified, {(time.time()-t0)/len(runs):.1f}s per question")

## The app (Gradio user interface)
**What it does:** builds a simple web app inside the notebook with four tabs:

| Tab | What you can do |
|---|---|
| **Dashboard** | see revenue, units, orders, products needing attention, and 3 charts |
| **Assistant** | chat with the agents; a live panel shows which agent is working |
| **Data** | upload your own CSV / Excel / PDF files (any common column names). Dashboard + Actions refresh automatically; see validation results and rejected rows with reasons. **Reset to sample data** goes back to the demo shop |
| **Actions** | review proposed actions, **approve / reject / complete** them, and read the draft messages (never sent) |

`SHARE_GRADIO=False` keeps the app private to your Colab session. If you set it to `True`, anyone with the link can open it – do not use real/sensitive data.

**You should see:** the app appears under the cell (and a link). Try the example questions!

In [ ]:
import gradio as gr
import plotly.graph_objects as go

def dashboard(days=30):
    if DB["sales"] is None: return "No valid sales data. Upload sales.csv in the Data tab.", go.Figure(), go.Figure(), go.Figure()
    end = DB["as_of"]; f = analyze_sales(end - pd.Timedelta(days=int(days) - 1), end, f"last {int(days)} days"); inv = analyze_inventory()
    avg_rating = f"{DB['feedback']['rating'].mean():.2f}" if DB["feedback"] is not None and len(DB["feedback"]) else "n/a"
    kpi = (f"### {BUSINESS} - {f['period']} (as of {end.date()})\n| Revenue | Units | Orders | Avg order | Products needing attention | Avg rating |\n|---|---|---|---|---|---|\n"
           f"| **{money(f['revenue'])}** | **{f['units']}** | **{f['orders']}** | **{money(f['aov'])}** | **{sum(i['risk'] in ('critical','high','medium') for i in inv['items'])}** | "
           f"**{avg_rating}** |")
    d = DB["sales"][DB["sales"]["date"] > end - pd.Timedelta(days=int(days))].groupby("date")["line_revenue"].sum()
    f1 = go.Figure(go.Scatter(x=d.index, y=d.values, mode="lines")); f1.update_layout(title="Daily revenue", height=300, margin=dict(t=40, b=10))
    f2 = go.Figure(go.Bar(x=[t["revenue"] for t in f["top"]][::-1], y=[t["name"] for t in f["top"]][::-1], orientation="h")); f2.update_layout(title="Top products by revenue", height=300, margin=dict(t=40, b=10))
    rows = [i for i in inv["items"] if i["coverage"] is not None][:8]; colors = {"critical": "#c0392b", "high": "#e67e22", "medium": "#f1c40f", "low": "#27ae60", "unknown": "#999"}
    f3 = go.Figure([go.Bar(x=[i["name"] for i in rows], y=[i["coverage"] for i in rows], marker_color=[colors[i["risk"]] for i in rows], name="Days of stock"),
                    go.Scatter(x=[i["name"] for i in rows], y=[i["lead_time"] for i in rows], mode="markers", marker=dict(symbol="diamond", size=10, color="black"), name="Supplier lead time")])
    f3.update_layout(title="Days of stock vs supplier lead time", height=300, margin=dict(t=40, b=10))
    return kpi, f1, f2, f3

def chat(message, history):
    history = list(history or []) + [{"role": "user", "content": message}]; lines = []       # show the question straight away
    yield history, "", "_working..._"
    for state in GRAPH.stream(new_state(message), stream_mode="values"):          # stream = see each agent finish
        lines = [f"{'OK ' if x['status'] == 'ok' else 'ERR'} **{x['agent']}** - {x['summary']} ({x['seconds']}s)" for x in state.get("steps", [])]
        yield history, "", "\n\n".join(lines) + "\n\n_working..._"
    history = history + [{"role": "assistant", "content": state.get("answer", "No answer.")}]
    yield history, "", "\n\n".join(lines) + f"\n\n**Finished: {state.get('status')}**"

def _data_tables():
    """Validation summary + rejected rows (with reasons) as tables for the Data tab."""
    rep = pd.DataFrame([{**{k: v for k, v in r.items() if k not in ("renamed", "columns_seen")}, "issues": str(r["issues"]),
                         "renamed_columns": ", ".join(f"{a} -> {b}" for a, b in r.get("renamed", {}).items())} for r in DB["reports"].values()])
    parts = [x.assign(dataset=k) for k, x in DB["rejected"].items() if len(x)]
    if not parts: return rep, pd.DataFrame()
    rej = pd.concat(parts, ignore_index=True)
    first = [c for c in ("dataset", "reject_reason", "reject_detail") if c in rej.columns]
    return rep, rej[first + [c for c in rej.columns if c not in first]]

def _after_data_change(note):
    """New data = old proposals, chat memory and answers no longer apply."""
    retired = retire_open_proposals(note); MEMORY.clear(); LAST_PLAN.clear(); NARRATE_CACHE.clear()
    return retired

def process_uploads(files):
    if not files: return "Choose one or more files first.", pd.DataFrame(), pd.DataFrame()
    notes, data_changed = [], False
    stems = {"sales": "sales", "inventory": "inventory", "feedback": "customer_feedback"}
    for f in files:
        name = os.path.basename(f)
        if name.lower().endswith(".pdf"):
            import shutil; shutil.copy(f, f"{DATA}/{name}"); notes.append(f"✅ **{name}**: stored as a document for the Document Researcher."); continue
        try: df = read_any(f)
        except Exception as e: notes.append(f"❌ **{name}**: could not read the file ({e}). Nothing was changed."); continue
        kind, score, missing = detect_kind(df)
        seen = ", ".join(map(str, df.columns))[:200]
        if score < 0.5:
            notes.append(f"❌ **{name}**: not recognised as sales, inventory or feedback data. Columns found: {seen}. Nothing was changed."); continue
        if missing:
            notes.append(f"❌ **{name}**: looks like **{kind}** data but these columns are missing: {', '.join(missing)}. Columns found: {seen}. Nothing was changed."); continue
        canon, renamed = canonicalize(df); canon = fill_ids(canon)
        clean, rejected, rep_ = validate(canon, kind)
        if not rep_["usable"]:
            notes.append(f"❌ **{name}**: all {rep_['rows']} rows were rejected {rep_['issues']}. Nothing was changed."); continue
        # Add new sales/feedback rows to the existing data. Inventory is an
        # upsert: uploaded product IDs update existing products instead of duplicating them.
        target = f"{DATA}/{stems[kind]}.csv"
        existing_path = next((f"{DATA}/{stems[kind]}{e}" for e in (".csv", ".xlsx", ".xls")
                              if os.path.exists(f"{DATA}/{stems[kind]}{e}")), None)
        incoming = canon.copy()
        if existing_path:
            existing = canonicalize(read_any(existing_path))[0]
            if kind == "inventory" and "product_id" in existing and "product_id" in incoming:
                merged = pd.concat([existing, incoming], ignore_index=True)
                merged = merged.drop_duplicates(subset=["product_id"], keep="last")
            else:
                merged = pd.concat([existing, incoming], ignore_index=True, sort=False)
                if kind == "feedback" and "feedback_id" in merged:
                    merged = merged.drop_duplicates(subset=["feedback_id"], keep="first")
            canon = merged
        # Store only the canonical CSV so load_data always reads the newest version.
        for e in (".csv", ".xlsx", ".xls"):
            p = f"{DATA}/{stems[kind]}{e}"
            if os.path.exists(p): os.remove(p)
        canon.to_csv(target, index=False)
        data_changed = True
        mp = ("; renamed columns: " + ", ".join(f"{a} → {b}" for a, b in renamed.items())) if renamed else ""
        hint = ""
        if rep_["rejected"] / max(rep_["rows"], 1) >= 0.3:
            hint = " ⚠️ **Many rows were rejected** - see the reasons in the table below."
            if kind == "feedback" and "bad_rating" in rep_["issues"]: hint += " If your ratings use a 1-10 scale, set `RATING_MAX = 10` in the Agent 1 cell and run again."
        notes.append(f"✅ **{name}**: stored as **{kind}** data - {rep_['clean']}/{rep_['rows']} rows valid, {rep_['rejected']} rejected{mp}.{hint}")
    retired = _after_data_change("superseded: new data uploaded") if data_changed else 0
    load_data(); n = build_index(DB["pdfs"])
    extra = [f"Reprocessed. Documents indexed: {n} chunks."]
    if data_changed:
        fresh = generate_proposals()
        extra.append(f"Dashboard and Actions were refreshed from your data: **{fresh} new proposed action(s)** are waiting in the Actions tab; {retired} old one(s) from the previous data were retired (marked rejected: 'superseded', kept in the history).")
    extra += [f"⚠️ {w}" for w in DB["warnings"] if "not found" not in w]
    return "\n\n".join(notes + [""] + extra), *_data_tables()

def reset_sample():
    # Restore the original CSV/PDF sample files and clear all app history so the
    # dashboard, actions, drafts, reports and run history start clean.
    for fn in os.listdir(DATA):
        path = f"{DATA}/{fn}"
        if os.path.isfile(path): os.remove(path)
    make_sample_data()
    MEMORY.clear(); LAST_PLAN.clear(); NARRATE_CACHE.clear()
    with conn() as c:
        for table in ("history", "actions", "runs", "drafts", "reports"):
            c.execute(f"DELETE FROM {table}")
    load_data(); n = build_index(DB["pdfs"]); fresh = generate_proposals()
    return f"✅ Reset complete. Default sample data restored; dashboard recalculated; {n} document chunks indexed; {fresh} fresh proposed action(s) generated.", *_data_tables()

def _rows_for(status):
    """The actions that belong to one value of the 'Show' dropdown ('open' = proposed + approved)."""
    rows = list_actions()
    if status == "open": return [r for r in rows if r["status"] in ("proposed", "approved")]
    return rows if status == "all" else [r for r in rows if r["status"] == status]

def action_table(status="open"):
    return pd.DataFrame([{"id": r["id"], "priority": r["priority"], "status": r["status"], "title": r["title"], "evidence": ", ".join(json.loads(r["evidence"])[:4]), "next step": r["next_step"]} for r in _rows_for(status)])

def action_choices(status="open"):
    """The 'Choose an action' list always matches the selected 'Show' view: same actions as the table."""
    return [f"{r['id']} | {r['title'][:50]}" for r in _rows_for(status)]

def refresh(f): return action_table(f), gr.Dropdown(choices=action_choices(f), value=None)

def decide(action_id, new_status, status_filter):
    aid = (action_id or "").split(" | ")[0].strip()
    if not aid: msg = "⚠️ Choose an action in the list first."
    else:
        try: set_status(aid, new_status, "changed in the app"); msg = f"✅ {aid} is now **{new_status}**."
        except ValueError as e:
            msg = f"⚠️ Not allowed: {e}." + (" Approve it first, then mark it completed." if new_status == "completed" else "")
        except KeyError: msg = f"⚠️ Unknown action {aid}."
    return action_table(status_filter), gr.Dropdown(choices=action_choices(status_filter), value=None), msg

def draft_table():
    with conn() as c: return pd.DataFrame([dict(r) for r in c.execute("SELECT id, kind, recipient, subject, status, body FROM drafts ORDER BY rowid DESC LIMIT 50")])

with gr.Blocks(title=f"{BUSINESS} AI Operations Manager") as demo:
    gr.Markdown(f"# {BUSINESS} - AI Operations Manager\nEight cooperating agents. Everything is a **proposal or draft**: nothing is emailed, ordered or sent.")
    with gr.Tabs():
        with gr.Tab("Dashboard"):
            days = gr.Dropdown([7, 14, 30, 60], value=30, label="Window (days)"); kpi = gr.Markdown()
            with gr.Row(): p1, p2 = gr.Plot(), gr.Plot()
            p3 = gr.Plot(); days.change(dashboard, days, [kpi, p1, p2, p3]); demo.load(dashboard, days, [kpi, p1, p2, p3])
        with gr.Tab("Assistant"):
            try: bot = gr.Chatbot(type="messages", height=420)
            except TypeError: bot = gr.Chatbot(height=420)
            progress = gr.Markdown("_Agent progress shows here._"); box = gr.Textbox(label="Your question", placeholder="Ask about sales, stock, feedback, documents, priorities or a report...")
            gr.Examples(QUESTIONS, box); box.submit(chat, [box, bot], [bot, box, progress])
            gr.Button("Send", variant="primary").click(chat, [box, bot], [bot, box, progress]); gr.Button("Clear memory").click(lambda: (MEMORY.clear(), [], "Cleared.")[1:], None, [bot, progress])
        with gr.Tab("Data"):
            up = gr.File(file_count="multiple", type="filepath", label="Add sales / feedback rows, update inventory products, or add PDFs"); msg = gr.Markdown()
            vr, rj = gr.Dataframe(label="Validation summary", interactive=False), gr.Dataframe(label="Rejected rows (with reason)", interactive=False)
            upload_btn = gr.Button("Store and reprocess", variant="primary"); reset_btn = gr.Button("Reset EVERYTHING to sample data", variant="stop")
        with gr.Tab("Actions"):
            flt = gr.Dropdown(["open", "all", "proposed", "approved", "rejected", "completed"], value="open", label="Show"); tbl = gr.Dataframe(interactive=False)
            sel = gr.Dropdown(choices=[], label="Choose an action"); note = gr.Markdown()
            with gr.Row():
                b1, b2, b3 = gr.Button("Approve", variant="primary"), gr.Button("Reject", variant="stop"), gr.Button("Mark completed")
            for b, st_ in ((b1, "approved"), (b2, "rejected"), (b3, "completed")): b.click(lambda a, f, st_=st_: decide(a, st_, f), [sel, flt], [tbl, sel, note])
            flt.change(refresh, flt, [tbl, sel]); gr.Button("Refresh").click(refresh, flt, [tbl, sel]); demo.load(refresh, flt, [tbl, sel])
            gr.Markdown("### Draft messages (never sent)"); dtbl = gr.Dataframe(interactive=False); gr.Button("Show drafts").click(draft_table, None, dtbl)

    # after any data change, refresh the Data tables AND the Dashboard AND the Actions tab (before, only the Data tab changed)
    for btn, fn, inp in ((upload_btn, process_uploads, up), (reset_btn, reset_sample, None)):
        btn.click(fn, inp, [msg, vr, rj]).then(dashboard, days, [kpi, p1, p2, p3]).then(refresh, flt, [tbl, sel])

demo.queue().launch(share=SHARE_GRADIO)

## Export (optional)
**What it does:** zips the working folder (data, reports, SQLite database) so you can download it before the Colab session ends.

In [ ]:
import shutil
zip_path = shutil.make_archive("/content/nova_simple_export", "zip", WORK); print("Created", zip_path)
try:
    from google.colab import files; files.download(zip_path)
except Exception: print("Download it from the Files panel on the left.")

## If something goes wrong
| Problem | What to do |
|---|---|
| Install errors / odd import errors | Runtime ▸ **Restart session**, then run from Step 2 (skip Step 1). |
| *CUDA out of memory* | Set `LOAD_IN_4BIT = True` in Step 2, restart, run again. |
| Model will not download | Check internet, re-run Step 5 (downloads resume). Or set `USE_LLM = False` – everything still works with plain wording. |
| No GPU | Works but slowly. Use Runtime ▸ Change runtime type ▸ T4 GPU. |
| Colab reset and files vanished | Run all cells again. To keep files, mount Google Drive and set `WORK` to a Drive folder in Step 3. |
| A cell is very slow | The AI is writing/classifying. Set `USE_NARRATION = False` and/or `FEEDBACK_USE_LLM = False` in Step 2 (instant), or keep them on - results are cached after the first run. |
| Upload says "Nothing was changed" | The message lists the missing columns and the columns it found. Rename the headers (or use the test files) and upload again. Upload sales + inventory + feedback **from the same shop** together. |
| Answers look empty | Open the **Data** tab: rows may have been rejected. Fix the file and upload it again. |
| An ❌ in the self-checks | Read the line: it names exactly what was expected. |

## How this meets the six project objectives
1. **Data retrieval** – Agent 1 loads/validates CSV, Excel, PDFs; Agent 5 retrieves PDF passages with page citations.
2. **LLM processing / summarisation** – Qwen classifies feedback, writes cited document answers and explains verified numbers.
3. **Automated actions** – Agent 8 writes reports/CSVs, drafts messages and manages approve/reject/complete.
4. **Orchestration** – LangGraph routes, runs agents in parallel, coordinates, verifies, repairs once, and saves history.
5. **Documentation / demo** – every cell is explained in plain language; the Assistant tab shows live agent activity (great for a face-cam demo).
6. **Testing / evaluation** – the *Self-checks* and *Evaluation* cells report only what they measure.

